In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## PrePost Changedetection DataLoader

In [2]:
import os
import random
import numpy as np
import torch

from pathlib import Path
from collections import OrderedDict
from torch.utils.data import Dataset, DataLoader


# ============================================================
# GLOBAL REPRODUCIBILITY
# ============================================================

GLOBAL_SEED = 42


def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducibility settings
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed(GLOBAL_SEED)


# ============================================================
# BASIC UTILITIES
# ============================================================

def normalize_image(image):
    """
    Convert uint8 image from [0, 255] to float32 [0, 1].

    This preserves the normalization used in the original
    STRATA implementation.
    """

    image = image.astype(np.float32) / 255.0
    return image


def ensure_channel(image):
    """
    Ensure that a grayscale image has a channel dimension.

    (H, W) -> (H, W, 1)
    """

    if image.ndim == 2:
        image = image[:, :, None]

    return image


def safe_copy(array):
    """
    Make the array contiguous after flips and rotations.
    """

    return np.ascontiguousarray(array)


def get_event_name(file_path):
    """
    Extract event name from an NPZ filename.

    Examples:
        EARTHQUAKE-TURKEY_000013_0_0.npz
            -> EARTHQUAKE-TURKEY

        TEXAS-TORNADOES_000006_256_512.npz
            -> TEXAS-TORNADOES
    """

    filename = os.path.basename(file_path)
    return filename.split("_")[0]

In [3]:
def joint_random_flip(
    pre_image,
    post_image,
    damage_mask,
):
    """
    Apply identical horizontal and vertical flips to
    pre-image, post-image, and damage mask.
    """

    # Horizontal flip
    if random.random() < 0.5:
        pre_image = pre_image[:, ::-1, :]
        post_image = post_image[:, ::-1, :]
        damage_mask = damage_mask[:, ::-1]

    # Vertical flip
    if random.random() < 0.5:
        pre_image = pre_image[::-1, :, :]
        post_image = post_image[::-1, :, :]
        damage_mask = damage_mask[::-1, :]

    return pre_image, post_image, damage_mask


def joint_random_rotate(
    pre_image,
    post_image,
    damage_mask,
):
    """
    Apply the same random 0/90/180/270-degree rotation
    to both temporal images and the label.
    """

    k = random.randint(0, 3)

    pre_image = np.rot90(
        pre_image,
        k=k,
        axes=(0, 1),
    )

    post_image = np.rot90(
        post_image,
        k=k,
        axes=(0, 1),
    )

    damage_mask = np.rot90(
        damage_mask,
        k=k,
        axes=(0, 1),
    )

    return pre_image, post_image, damage_mask

In [4]:
def paired_brightness_scaling(
    pre_image,
    post_image,
    probability=0.5,
    minimum_factor=0.8,
    maximum_factor=1.2,
):
    """
    Apply the same brightness factor to pre and post images.

    Using the same factor avoids creating artificial
    temporal differences.
    """

    if random.random() < probability:

        factor = random.uniform(
            minimum_factor,
            maximum_factor,
        )

        pre_image = pre_image * factor
        post_image = post_image * factor

    pre_image = np.clip(
        pre_image,
        0.0,
        1.0,
    )

    post_image = np.clip(
        post_image,
        0.0,
        1.0,
    )

    return pre_image, post_image


def paired_gaussian_noise(
    pre_image,
    post_image,
    probability=0.3,
    noise_std=0.02,
):
    """
    Apply mild Gaussian noise to both temporal images.

    The same noise field is used for both images to avoid
    creating artificial change evidence.
    """

    if random.random() < probability:

        noise = np.random.normal(
            loc=0.0,
            scale=noise_std,
            size=pre_image.shape,
        ).astype(np.float32)

        pre_image = pre_image + noise
        post_image = post_image + noise

    pre_image = np.clip(
        pre_image,
        0.0,
        1.0,
    )

    post_image = np.clip(
        post_image,
        0.0,
        1.0,
    )

    return pre_image, post_image

In [5]:
class NPZCache:
    """
    Small per-worker in-memory cache.

    A limited cache is used because each paired sample contains
    two RGB images and one mask.
    """

    def __init__(self, max_size=128):

        self.cache = OrderedDict()
        self.max_size = max_size

    def get(self, file_path):

        file_path = str(file_path)

        if file_path in self.cache:
            self.cache.move_to_end(file_path)
            return self.cache[file_path]

        # The context manager closes the NPZ file immediately
        # after its arrays are copied.
        with np.load(file_path) as data:

            sample = {
                "pre_image": data["pre_image"].copy(),
                "post_image": data["post_image"].copy(),
                "damage_mask": data["damage_mask"].copy(),
            }

        if len(self.cache) >= self.max_size:
            self.cache.popitem(last=False)

        self.cache[file_path] = sample

        return sample

In [6]:
class PairedBuildingDamageDataset(Dataset):
    """
    Supervised paired pre/post dataset for bi-temporal
    building-damage segmentation.

    Each item returns:
        pre_image:   FloatTensor [3, H, W]
        post_image:  FloatTensor [3, H, W]
        damage_mask: LongTensor  [H, W]
        valid_mask:  BoolTensor  [H, W]
        event_name:  str

    Label definition:
        0 = Background
        1 = Intact
        2 = Damage
        3 = Destroyed
    """

    def __init__(
        self,
        root_dir=None,
        files=None,
        augment=False,
        cache_size=128,
    ):

        # ----------------------------------------------------
        # Use an explicitly supplied split whenever available
        # ----------------------------------------------------

        if files is not None:

            self.files = sorted([
                str(file_path)
                for file_path in files
            ])

        elif root_dir is not None:

            root_dir = Path(root_dir)

            self.files = sorted([
                str(file_path)
                for file_path in root_dir.glob("*.npz")
            ])

        else:
            raise ValueError(
                "Either root_dir or files must be provided."
            )

        if len(self.files) == 0:
            raise RuntimeError(
                "No NPZ files were found for this dataset."
            )

        self.augment = augment
        self.cache = NPZCache(
            max_size=cache_size
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):

        file_path = self.files[index]
        sample = self.cache.get(file_path)

        pre_image = sample["pre_image"].copy()
        post_image = sample["post_image"].copy()
        damage_mask = sample["damage_mask"].copy()

        # ----------------------------------------------------
        # Basic validation
        # ----------------------------------------------------

        pre_image = ensure_channel(pre_image)
        post_image = ensure_channel(post_image)

        if pre_image.shape != post_image.shape:
            raise ValueError(
                f"Pre/post shape mismatch in {file_path}: "
                f"{pre_image.shape} versus {post_image.shape}"
            )

        if pre_image.shape[:2] != damage_mask.shape:
            raise ValueError(
                f"Image/mask shape mismatch in {file_path}: "
                f"{pre_image.shape[:2]} versus "
                f"{damage_mask.shape}"
            )

        unique_labels = set(
            np.unique(damage_mask).tolist()
        )

        if not unique_labels.issubset({0, 1, 2, 3}):
            raise ValueError(
                f"Unexpected labels in {file_path}: "
                f"{sorted(unique_labels)}"
            )

        # ----------------------------------------------------
        # Normalization: same [0,1] convention as STRATA
        # ----------------------------------------------------

        pre_image = normalize_image(pre_image)
        post_image = normalize_image(post_image)

        # ----------------------------------------------------
        # Joint paired augmentation
        # ----------------------------------------------------

        if self.augment:

            pre_image, post_image, damage_mask = (
                joint_random_flip(
                    pre_image,
                    post_image,
                    damage_mask,
                )
            )

            pre_image, post_image, damage_mask = (
                joint_random_rotate(
                    pre_image,
                    post_image,
                    damage_mask,
                )
            )

            # Use stronger appearance augmentation for patches
            # containing Damage or Destroyed buildings.
            contains_rare_damage = np.any(
                (damage_mask == 2) |
                (damage_mask == 3)
            )

            if contains_rare_damage:

                pre_image, post_image = (
                    paired_gaussian_noise(
                        pre_image,
                        post_image,
                        probability=0.3,
                        noise_std=0.02,
                    )
                )

                pre_image, post_image = (
                    paired_brightness_scaling(
                        pre_image,
                        post_image,
                        probability=0.5,
                    )
                )

            else:

                pre_image, post_image = (
                    paired_gaussian_noise(
                        pre_image,
                        post_image,
                        probability=0.15,
                        noise_std=0.02,
                    )
                )

                pre_image, post_image = (
                    paired_brightness_scaling(
                        pre_image,
                        post_image,
                        probability=0.3,
                    )
                )

        # ----------------------------------------------------
        # Building-only valid mask
        #
        # A separate building mask is unnecessary because:
        #   0 = background
        #   1, 2, 3 = building damage classes
        # ----------------------------------------------------

        valid_mask = damage_mask > 0

        # ----------------------------------------------------
        # Safe contiguous arrays after flip/rotation
        # ----------------------------------------------------

        pre_image = safe_copy(pre_image)
        post_image = safe_copy(post_image)
        damage_mask = safe_copy(damage_mask)
        valid_mask = safe_copy(valid_mask)

        # ----------------------------------------------------
        # Convert HWC -> CHW
        # ----------------------------------------------------

        pre_image = np.transpose(
            pre_image,
            (2, 0, 1),
        )

        post_image = np.transpose(
            post_image,
            (2, 0, 1),
        )

        # ----------------------------------------------------
        # Convert to tensors
        # ----------------------------------------------------

        pre_image = torch.from_numpy(
            pre_image
        ).float()

        post_image = torch.from_numpy(
            post_image
        ).float()

        damage_mask = torch.from_numpy(
            damage_mask
        ).long()

        valid_mask = torch.from_numpy(
            valid_mask
        ).bool()

        event_name = get_event_name(
            file_path
        )

        return (
            pre_image,
            post_image,
            damage_mask,
            valid_mask,
            event_name,
        )

In [7]:
def seed_dataloader_worker(worker_id):
    """
    Give every DataLoader worker a deterministic but distinct seed.
    """

    worker_seed = (
        torch.initial_seed() % (2**32)
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def get_paired_loader(
    dataset,
    batch_size=16,
    shuffle=True,
    sampler=None,
    drop_last=False,
    num_workers=4,
    pin_memory=True,
):

    # A sampler and shuffle=True cannot be used together.
    if sampler is not None:
        shuffle = False

    generator = torch.Generator()
    generator.manual_seed(GLOBAL_SEED)

    loader_arguments = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "sampler": sampler,
        "drop_last": drop_last,
        "num_workers": num_workers,
        "pin_memory": pin_memory,
        "worker_init_fn": seed_dataloader_worker,
        "generator": generator,
    }

    # These arguments are only valid when workers are enabled.
    if num_workers > 0:
        loader_arguments.update({
            "persistent_workers": True,
            "prefetch_factor": 2,
        })

    return DataLoader(**loader_arguments)

In [8]:
DATA_ROOT = (
    "/content/ebd_target_prepost_npz"
)

In [9]:
import os
import zipfile

ZIP_PATH = (
    "/content/drive/MyDrive/"
    "EBD_Target_PrePost_256.zip"
)

DATA_ROOT = (
    "/content/ebd_target_prepost_npz"
)

os.makedirs(
    DATA_ROOT,
    exist_ok=True,
)

extraction_marker = os.path.join(
    DATA_ROOT,
    ".extraction_complete",
)

if not os.path.exists(extraction_marker):

    print("Extracting paired dataset...")

    with zipfile.ZipFile(
        ZIP_PATH,
        "r",
    ) as zip_ref:
        zip_ref.extractall(DATA_ROOT)

    Path(extraction_marker).touch()

    print("Extraction completed.")

else:
    print("Dataset was already extracted.")

Extracting paired dataset...
Extraction completed.


In [10]:
full_dataset = PairedBuildingDamageDataset(
    root_dir=DATA_ROOT,
    augment=False,
    cache_size=32,
)

print("Dataset size:", len(full_dataset))

sample = full_dataset[0]

pre_image, post_image, damage_mask, valid_mask, event_name = sample

print("Event:", event_name)
print("Pre image:", pre_image.shape, pre_image.dtype)
print("Post image:", post_image.shape, post_image.dtype)
print("Damage mask:", damage_mask.shape, damage_mask.dtype)
print("Valid mask:", valid_mask.shape, valid_mask.dtype)

print(
    "Damage classes:",
    torch.unique(damage_mask).tolist(),
)

print(
    "Valid building pixels:",
    valid_mask.sum().item(),
)

print(
    "Pre range:",
    pre_image.min().item(),
    pre_image.max().item(),
)

print(
    "Post range:",
    post_image.min().item(),
    post_image.max().item(),
)

Dataset size: 13105
Event: EARTHQUAKE-TURKEY
Pre image: torch.Size([3, 256, 256]) torch.float32
Post image: torch.Size([3, 256, 256]) torch.float32
Damage mask: torch.Size([256, 256]) torch.int64
Valid mask: torch.Size([256, 256]) torch.bool
Damage classes: [0, 1]
Valid building pixels: 1936
Pre range: 0.0235294122248888 1.0
Post range: 0.0 1.0


## Model Architecture

In [11]:
# ============================================================
# ChangeOS-R18
# Trainable reconstruction of the official ChangeOS structure
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as TF

from torchvision.models import (
    resnet18,
    ResNet18_Weights,
)


# ============================================================
# 1. SQUEEZE-AND-EXCITATION BLOCK
# ============================================================

class SEBlock(nn.Module):

    def __init__(
        self,
        channels,
        reduction=16,
    ):
        super().__init__()

        hidden_channels = max(
            channels // reduction,
            1,
        )

        self.global_pool = nn.AdaptiveAvgPool2d(
            output_size=1
        )

        self.channel_gate = nn.Sequential(
            nn.Linear(
                channels,
                hidden_channels,
            ),
            nn.ReLU(
                inplace=True
            ),
            nn.Linear(
                hidden_channels,
                channels,
            ),
            nn.Sigmoid(),
        )

    def forward(
        self,
        features,
    ):
        batch_size, channels, _, _ = (
            features.shape
        )

        weights = self.global_pool(
            features
        ).view(
            batch_size,
            channels,
        )

        weights = self.channel_gate(
            weights
        ).view(
            batch_size,
            channels,
            1,
            1,
        )

        return features * weights


# ============================================================
# 2. FPN CONVOLUTION BLOCK
# ============================================================

class FPNConvBlock(nn.Module):
    """
    Official FPN ConvBlock:
        Conv2d -> BatchNorm

    No activation is applied in the official exported model.
    """

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        padding=0,
    ):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False,
            ),
            nn.BatchNorm2d(
                out_channels
            ),
        )

    def forward(
        self,
        features,
    ):
        return self.block(
            features
        )


# ============================================================
# 3. FEATURE PYRAMID NETWORK
# ============================================================

class ChangeOSFPN(nn.Module):
    """
    Official ChangeOS FPN for ResNet-18 features.

    Input order:
        c5: [B, 512, H/32, W/32]
        c4: [B, 256, H/16, W/16]
        c3: [B, 128, H/8,  W/8]
        c2: [B,  64, H/4,  W/4]

    Output:
        p2, p3, p4, p5, each with 256 channels
    """

    def __init__(
        self,
        out_channels=256,
    ):
        super().__init__()

        self.inner_c2 = FPNConvBlock(
            64,
            out_channels,
            kernel_size=1,
        )

        self.inner_c3 = FPNConvBlock(
            128,
            out_channels,
            kernel_size=1,
        )

        self.inner_c4 = FPNConvBlock(
            256,
            out_channels,
            kernel_size=1,
        )

        self.inner_c5 = FPNConvBlock(
            512,
            out_channels,
            kernel_size=1,
        )

        self.output_p2 = FPNConvBlock(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
        )

        self.output_p3 = FPNConvBlock(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
        )

        self.output_p4 = FPNConvBlock(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
        )

        self.output_p5 = FPNConvBlock(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
        )

    @staticmethod
    def upsample_nearest(
        features,
        target_features,
    ):
        return TF.interpolate(
            features,
            size=target_features.shape[-2:],
            mode="nearest",
        )

    def forward(
        self,
        c5,
        c4,
        c3,
        c2,
    ):
        inner_p5 = self.inner_c5(
            c5
        )

        inner_p4 = (
            self.inner_c4(c4)
            + self.upsample_nearest(
                inner_p5,
                c4,
            )
        )

        inner_p3 = (
            self.inner_c3(c3)
            + self.upsample_nearest(
                inner_p4,
                c3,
            )
        )

        inner_p2 = (
            self.inner_c2(c2)
            + self.upsample_nearest(
                inner_p3,
                c2,
            )
        )

        p2 = self.output_p2(
            inner_p2
        )

        p3 = self.output_p3(
            inner_p3
        )

        p4 = self.output_p4(
            inner_p4
        )

        p5 = self.output_p5(
            inner_p5
        )

        return p2, p3, p4, p5


# ============================================================
# 4. ASYMMETRIC DECODER BLOCK
# ============================================================

class ConvBNReLUUpsample(nn.Module):

    def __init__(
        self,
        channels=256,
        upsample=False,
    ):
        super().__init__()

        layers = [
            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(
                channels
            ),
            nn.ReLU(
                inplace=True
            ),
        ]

        if upsample:
            # Matches nn.UpsamplingBilinear2d in the JIT model
            layers.append(
                nn.UpsamplingBilinear2d(
                    scale_factor=2
                )
            )

        self.block = nn.Sequential(
            *layers
        )

    def forward(
        self,
        features,
    ):
        return self.block(
            features
        )


class AsymmetricDecoder(nn.Module):
    """
    Brings all FPN levels to the P2 spatial resolution
    and averages them.

    P2: one convolution
    P3: one convolution + one upsampling
    P4: two convolution/upsampling blocks
    P5: three convolution/upsampling blocks
    """

    def __init__(
        self,
        channels=256,
    ):
        super().__init__()

        self.p2_path = nn.Sequential(
            ConvBNReLUUpsample(
                channels,
                upsample=False,
            )
        )

        self.p3_path = nn.Sequential(
            ConvBNReLUUpsample(
                channels,
                upsample=True,
            )
        )

        self.p4_path = nn.Sequential(
            ConvBNReLUUpsample(
                channels,
                upsample=True,
            ),
            ConvBNReLUUpsample(
                channels,
                upsample=True,
            ),
        )

        self.p5_path = nn.Sequential(
            ConvBNReLUUpsample(
                channels,
                upsample=True,
            ),
            ConvBNReLUUpsample(
                channels,
                upsample=True,
            ),
            ConvBNReLUUpsample(
                channels,
                upsample=True,
            ),
        )

    def forward(
        self,
        p2,
        p3,
        p4,
        p5,
    ):
        decoded_p2 = self.p2_path(
            p2
        )

        decoded_p3 = self.p3_path(
            p3
        )

        decoded_p4 = self.p4_path(
            p4
        )

        decoded_p5 = self.p5_path(
            p5
        )

        decoded_features = (
            decoded_p2
            + decoded_p3
            + decoded_p4
            + decoded_p5
        ) / 4.0

        return decoded_features


# ============================================================
# 5. FPN + ASYMMETRIC DECODER NECK
# ============================================================

class ChangeOSNeck(nn.Module):

    def __init__(
        self,
        channels=256,
    ):
        super().__init__()

        self.fpn = ChangeOSFPN(
            out_channels=channels
        )

        self.decoder = AsymmetricDecoder(
            channels=channels
        )

    def forward(
        self,
        c5,
        c4,
        c3,
        c2,
    ):
        pyramid_features = self.fpn(
            c5,
            c4,
            c3,
            c2,
        )

        return self.decoder(
            *pyramid_features
        )


# ============================================================
# 6. FUSION MODULE
# ============================================================

class FuseConv(nn.Module):
    """
    Official ChangeOS fusion:
        concat(localization, damage)
        -> 1x1 Conv
        -> BN
        -> ReLU
        -> SE residual
        -> ReLU
    """

    def __init__(
        self,
        channels=256,
    ):
        super().__init__()

        self.conv = nn.Conv2d(
            channels * 2,
            channels,
            kernel_size=1,
            bias=True,
        )

        self.batch_norm = nn.BatchNorm2d(
            channels
        )

        self.relu = nn.ReLU(
            inplace=True
        )

        self.se = SEBlock(
            channels=channels,
            reduction=16,
        )

    def forward(
        self,
        concatenated_features,
    ):
        features = self.conv(
            concatenated_features
        )

        features = self.batch_norm(
            features
        )

        features = self.relu(
            features
        )

        features = (
            features
            + self.se(features)
        )

        features = self.relu(
            features
        )

        return features


# ============================================================
# 7. DEEP PREDICTION HEAD
# ============================================================

class DeepHeadResidualBlock(nn.Module):

    def __init__(
        self,
        channels=256,
    ):
        super().__init__()

        middle_channels = (
            channels // 2
        )

        self.block = nn.Sequential(
            nn.Conv2d(
                channels,
                middle_channels,
                kernel_size=1,
                bias=True,
            ),
            nn.BatchNorm2d(
                middle_channels
            ),
            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                middle_channels,
                middle_channels,
                kernel_size=3,
                padding=1,
                bias=True,
            ),
            nn.BatchNorm2d(
                middle_channels
            ),

            nn.Conv2d(
                middle_channels,
                channels,
                kernel_size=1,
                bias=True,
            ),
            nn.BatchNorm2d(
                channels
            ),

            SEBlock(
                channels=channels,
                reduction=16,
            ),
        )

    def forward(
        self,
        features,
    ):
        return self.block(
            features
        )


class ChangeOSDeepHead(nn.Module):

    def __init__(
        self,
        out_channels,
        channels=256,
        output_scale=4,
    ):
        super().__init__()

        self.residual_block = (
            DeepHeadResidualBlock(
                channels=channels
            )
        )

        self.relu = nn.ReLU(
            inplace=True
        )

        self.classifier = nn.Conv2d(
            channels,
            out_channels,
            kernel_size=1,
            bias=True,
        )

        self.output_scale = (
            output_scale
        )

    def forward(
        self,
        features,
        residual_features=None,
        output_size=None,
    ):
        if residual_features is None:
            residual_features = features

        features = (
            self.residual_block(features)
            + residual_features
        )

        features = self.relu(
            features
        )

        logits = self.classifier(
            features
        )

        if output_size is not None:
            logits = TF.interpolate(
                logits,
                size=output_size,
                mode="bilinear",
                align_corners=True,
            )

        else:
            logits = TF.interpolate(
                logits,
                scale_factor=self.output_scale,
                mode="bilinear",
                align_corners=True,
            )

        return logits


# ============================================================
# 8. SHARED SIAMESE RESNET-18 ENCODER
# ============================================================

class SiameseResNet18Encoder(nn.Module):
    """
    Pre and post images are concatenated along the batch axis,
    matching the official ChangeOS implementation.

    This guarantees:
      - shared ResNet weights;
      - a single backbone forward;
      - joint BatchNorm statistics for both dates.
    """

    def __init__(
        self,
        imagenet_pretrained=False,
    ):
        super().__init__()

        weights = (
            ResNet18_Weights.DEFAULT
            if imagenet_pretrained
            else None
        )

        backbone = resnet18(
            weights=weights
        )

        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

    def forward(
        self,
        pre_image,
        post_image,
    ):
        if pre_image.shape != post_image.shape:
            raise ValueError(
                "Pre- and post-event tensors must have "
                "identical shapes."
            )

        batch_size = pre_image.shape[0]

        # Equivalent to the reshape performed in the official JIT
        combined_batch = torch.cat(
            [
                pre_image,
                post_image,
            ],
            dim=0,
        )

        features = self.conv1(
            combined_batch
        )

        features = self.bn1(
            features
        )

        features = self.relu(
            features
        )

        features = self.maxpool(
            features
        )

        c2 = self.layer1(
            features
        )

        c3 = self.layer2(
            c2
        )

        c4 = self.layer3(
            c3
        )

        c5 = self.layer4(
            c4
        )

        pre_c2, post_c2 = torch.split(
            c2,
            batch_size,
            dim=0,
        )

        pre_c3, post_c3 = torch.split(
            c3,
            batch_size,
            dim=0,
        )

        pre_c4, post_c4 = torch.split(
            c4,
            batch_size,
            dim=0,
        )

        pre_c5, post_c5 = torch.split(
            c5,
            batch_size,
            dim=0,
        )

        pre_features = (
            pre_c5,
            pre_c4,
            pre_c3,
            pre_c2,
        )

        post_features = (
            post_c5,
            post_c4,
            post_c3,
            post_c2,
        )

        return (
            pre_features,
            post_features,
        )


# ============================================================
# 9. OFFICIAL INPUT NORMALIZATION ADAPTER
# ============================================================

class ChangeOSInputAdapter(nn.Module):
    """
    The current shared DataLoader returns:
        BGR float images in [0, 1].

    The official ChangeOS wrapper expects:
        RGB images in [0, 255], normalized using:
        mean = [123.675, 116.28, 103.53]
        std  = [58.395, 57.12, 57.375]

    Dividing mean/std by 255 makes the transformation
    mathematically equivalent for [0,1] tensors.
    """

    def __init__(
        self,
        input_is_bgr=True,
    ):
        super().__init__()

        self.input_is_bgr = (
            input_is_bgr
        )

        rgb_mean = torch.tensor(
            [
                123.675 / 255.0,
                116.280 / 255.0,
                103.530 / 255.0,
            ],
            dtype=torch.float32,
        ).view(
            1,
            3,
            1,
            1,
        )

        rgb_std = torch.tensor(
            [
                58.395 / 255.0,
                57.120 / 255.0,
                57.375 / 255.0,
            ],
            dtype=torch.float32,
        ).view(
            1,
            3,
            1,
            1,
        )

        self.register_buffer(
            "rgb_mean",
            rgb_mean,
            persistent=True,
        )

        self.register_buffer(
            "rgb_std",
            rgb_std,
            persistent=True,
        )

    def forward(
        self,
        image,
    ):
        if image.ndim != 4:
            raise ValueError(
                f"Expected BCHW input, received {image.shape}."
            )

        if image.shape[1] != 3:
            raise ValueError(
                f"Expected three channels, received {image.shape[1]}."
            )

        # cv2 BGR -> official ChangeOS RGB
        if self.input_is_bgr:
            image = image[
                :,
                [2, 1, 0],
                :,
                :,
            ]

        image = (
            image - self.rgb_mean
        ) / self.rgb_std

        return image


# ============================================================
# 10. COMPLETE CHANGEOS-R18
# ============================================================

class ChangeOSR18(nn.Module):
    """
    Trainable ChangeOS-R18.

    Inputs
    ------
    pre_image:
        [B, 3, H, W], BGR, float in [0,1]

    post_image:
        [B, 3, H, W], BGR, float in [0,1]

    Outputs
    -------
    localization_logits:
        [B, 1, H, W]

        Binary building localization logit.
        Threshold zero during inference.

    damage_logits:
        [B, 4, H, W]

        0 = Background
        1 = Intact
        2 = Damage
        3 = Destroyed
    """

    def __init__(
        self,
        damage_classes=4,
        imagenet_pretrained=False,
        input_is_bgr=True,
        decoder_channels=256,
    ):
        super().__init__()

        if damage_classes != 4:
            raise ValueError(
                "This experiment requires exactly four "
                "harmonized damage classes."
            )

        self.damage_classes = (
            damage_classes
        )

        self.input_adapter = (
            ChangeOSInputAdapter(
                input_is_bgr=input_is_bgr
            )
        )

        self.encoder = (
            SiameseResNet18Encoder(
                imagenet_pretrained=
                    imagenet_pretrained
            )
        )

        # Separate official necks
        self.localization_neck = ChangeOSNeck(
            channels=decoder_channels
        )

        self.damage_neck = ChangeOSNeck(
            channels=decoder_channels
        )

        self.fusion = FuseConv(
            channels=decoder_channels
        )

        # Official localization head has one binary logit
        self.localization_head = ChangeOSDeepHead(
            out_channels=1,
            channels=decoder_channels,
            output_scale=4,
        )

        # Adapted from official five classes to our four classes
        self.damage_head = ChangeOSDeepHead(
            out_channels=damage_classes,
            channels=decoder_channels,
            output_scale=4,
        )

        self._initialize_new_layers()

    def _initialize_new_layers(
        self,
    ):
        """
        Initialize decoder and heads without overwriting
        an optional ImageNet-pretrained backbone.
        """
        new_modules = [
            self.localization_neck,
            self.damage_neck,
            self.fusion,
            self.localization_head,
            self.damage_head,
        ]

        for parent_module in new_modules:
            for module in parent_module.modules():

                if isinstance(
                    module,
                    nn.Conv2d,
                ):
                    nn.init.kaiming_normal_(
                        module.weight,
                        mode="fan_out",
                        nonlinearity="relu",
                    )

                    if module.bias is not None:
                        nn.init.zeros_(
                            module.bias
                        )

                elif isinstance(
                    module,
                    nn.BatchNorm2d,
                ):
                    nn.init.ones_(
                        module.weight
                    )

                    nn.init.zeros_(
                        module.bias
                    )

                elif isinstance(
                    module,
                    nn.Linear,
                ):
                    nn.init.normal_(
                        module.weight,
                        mean=0.0,
                        std=0.01,
                    )

                    if module.bias is not None:
                        nn.init.zeros_(
                            module.bias
                        )

    def forward(
        self,
        pre_image,
        post_image,
    ):
        if pre_image.ndim != 4:
            raise ValueError(
                "pre_image must have shape [B,C,H,W]."
            )

        if post_image.ndim != 4:
            raise ValueError(
                "post_image must have shape [B,C,H,W]."
            )

        if pre_image.shape != post_image.shape:
            raise ValueError(
                "Pre/post input shapes are different: "
                f"{pre_image.shape} vs {post_image.shape}"
            )

        output_size = pre_image.shape[-2:]

        pre_image = self.input_adapter(
            pre_image
        )

        post_image = self.input_adapter(
            post_image
        )

        (
            pre_features,
            post_features,
        ) = self.encoder(
            pre_image,
            post_image,
        )

        localization_features = (
            self.localization_neck(
                *pre_features
            )
        )

        damage_features = (
            self.damage_neck(
                *post_features
            )
        )

        fused_features = self.fusion(
            torch.cat(
                [
                    localization_features,
                    damage_features,
                ],
                dim=1,
            )
        )

        # The official JIT passes localization features twice:
        # residual_block(loc_features) + loc_features
        localization_logits = (
            self.localization_head(
                features=
                    localization_features,

                residual_features=
                    localization_features,

                output_size=
                    output_size,
            )
        )

        damage_logits = (
            self.damage_head(
                features=
                    fused_features,

                residual_features=
                    fused_features,

                output_size=
                    output_size,
            )
        )

        return (
            localization_logits,
            damage_logits,
        )


# ============================================================
# 11. PARAMETER COUNTER
# ============================================================

def count_model_parameters(
    model,
):
    total_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    return (
        total_parameters,
        trainable_parameters,
    )

In [16]:
# ============================================================
# TEST 1 — DUMMY FORWARD TEST
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

changeos_model = ChangeOSR18(
    damage_classes=4,

    # فعلاً برای تست معماری؛ هنگام تنظیم آموزش نهایی
    # درباره ImageNet initialization تصمیم می‌گیریم.
    imagenet_pretrained=False,

    # داده‌های NPZ با cv2 و به‌صورت BGR ذخیره شده‌اند.
    input_is_bgr=True,

    decoder_channels=256,
).to(DEVICE)

dummy_pre = torch.rand(
    2,
    3,
    256,
    256,
    device=DEVICE,
)

dummy_post = torch.rand(
    2,
    3,
    256,
    256,
    device=DEVICE,
)

changeos_model.eval()

with torch.inference_mode():
    (
        dummy_localization_logits,
        dummy_damage_logits,
    ) = changeos_model(
        dummy_pre,
        dummy_post,
    )

total_parameters, trainable_parameters = (
    count_model_parameters(
        changeos_model
    )
)

print("=" * 72)
print("CHANGEOS-R18 DUMMY FORWARD TEST")
print("=" * 72)
print("Device                 :", DEVICE)
print("Pre input              :", tuple(dummy_pre.shape))
print("Post input             :", tuple(dummy_post.shape))
print(
    "Localization logits    :",
    tuple(dummy_localization_logits.shape),
)
print(
    "Damage logits          :",
    tuple(dummy_damage_logits.shape),
)
print(
    "Total parameters       :",
    f"{total_parameters:,}",
)
print(
    "Trainable parameters   :",
    f"{trainable_parameters:,}",
)
print("=" * 72)

assert dummy_localization_logits.shape == (
    2,
    1,
    256,
    256,
)

assert dummy_damage_logits.shape == (
    2,
    4,
    256,
    256,
)

assert torch.isfinite(
    dummy_localization_logits
).all()

assert torch.isfinite(
    dummy_damage_logits
).all()

print("\n✅ ChangeOS-R18 dummy test passed.")

CHANGEOS-R18 DUMMY FORWARD TEST
Device                 : cuda
Pre input              : (2, 3, 256, 256)
Post input             : (2, 3, 256, 256)
Localization logits    : (2, 1, 256, 256)
Damage logits          : (2, 4, 256, 256)
Total parameters       : 25,247,093
Trainable parameters   : 25,247,093

✅ ChangeOS-R18 dummy test passed.


## Loss & Training

In [17]:
# ============================================================
# CHANGEOS-R18 CONFIGURATION
# ============================================================

from pathlib import Path
import torch


# ------------------------------------------------------------
# Experiment identity
# ------------------------------------------------------------

MODEL_NAME = "ChangeOS_R18"
RUN_NAME = "budget500_seed42"

TARGET_LABELS_PER_CLASS = 500
SEED = 42


# ------------------------------------------------------------
# Data and existing STRATA split
# ------------------------------------------------------------

PAIRED_DATA_ROOT = Path(
    "/content/ebd_target_prepost_npz"
)

EXPERIMENT_SPLIT_PATH = Path(
    "/content/drive/MyDrive/STRATA_runs/splits/"
    "ebd_npz_event_EQTURKEY_TEXASTORNADO_"
    "labeled_unlabeled_500.npz"
)


# ------------------------------------------------------------
# Output workspace
# ------------------------------------------------------------

PREPOST_WORKSPACE_ROOT = Path(
    "/content/drive/MyDrive/STRATA_runs/"
    "PrePost_Baselines"
)

OUTPUT_DIR = (
    PREPOST_WORKSPACE_ROOT /
    MODEL_NAME /
    RUN_NAME
)

MODEL_DIR = (
    OUTPUT_DIR /
    "models"
)

RESULTS_DIR = (
    OUTPUT_DIR /
    "results"
)

HISTORY_DIR = (
    OUTPUT_DIR /
    "history"
)

BEST_MODEL_PATH = (
    MODEL_DIR /
    "best_changeos_r18.pth"
)

LATEST_MODEL_PATH = (
    MODEL_DIR /
    "latest_changeos_r18.pth"
)

FINAL_EXCEL_PATH = (
    RESULTS_DIR /
    "final_validation_global_and_event_metrics.xlsx"
)

FINAL_TEXT_PATH = (
    RESULTS_DIR /
    "final_validation_summary.txt"
)

HISTORY_PATH = (
    HISTORY_DIR /
    "training_history.pth"
)

HISTORY_EXCEL_PATH = (
    HISTORY_DIR /
    "training_history.xlsx"
)


# ------------------------------------------------------------
# Create all required directories
# ------------------------------------------------------------

for directory in [
    OUTPUT_DIR,
    MODEL_DIR,
    RESULTS_DIR,
    HISTORY_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# Model configuration
# ------------------------------------------------------------

NUM_LOCALIZATION_CHANNELS = 1
NUM_DAMAGE_CLASSES = 4

CHANGEOS_DECODER_CHANNELS = 256

# False keeps the controlled experiment free from
# external pretrained representations.
#
# This is different from loading the official ChangeOS
# damage checkpoint, which must never be used here.
IMAGENET_PRETRAINED = False

# NPZ images were read and saved by OpenCV as BGR.
# ChangeOSInputAdapter converts them internally to RGB.
INPUT_IS_BGR = True


# ------------------------------------------------------------
# Joint ChangeOS training
# ------------------------------------------------------------

EPOCHS = 100

EVAL_EVERY = 3
EARLY_STOP_PATIENCE = 5

# ChangeOS-R18 has two 256-channel FPN branches.
# This setting is safer than batch size 64.
TRAIN_BATCH_SIZE = 16
VAL_BATCH_SIZE = 16

# Effective batch size:
# 16 × 4 = 64
GRADIENT_ACCUMULATION_STEPS = 4

NUM_WORKERS = 0
MAX_GRAD_NORM = 5.0

PIN_MEMORY = torch.cuda.is_available()
USE_AMP = torch.cuda.is_available()


# ------------------------------------------------------------
# Optimizer and scheduler
# ------------------------------------------------------------

LEARNING_RATE = 1e-3
MIN_LEARNING_RATE = 1e-6

WEIGHT_DECAY = 0.0

SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 2


# ------------------------------------------------------------
# ChangeOS joint loss configuration
# ------------------------------------------------------------

# ChangeOS localization produces one binary logit.
# Therefore BCEWithLogitsLoss is required.
LOCALIZATION_POS_WEIGHT = 15.0

# Same harmonized four-class imbalance treatment used in
# the controlled pre/post experiment.
#
# These are experiment weights; they are not claimed as
# official ChangeOS loss weights.
DAMAGE_CLASS_WEIGHTS = [
    1.0,    # 0: Background
    35.0,   # 1: Intact
    150.0,  # 2: Damage
    120.0,  # 3: Destroyed
]

# Joint multi-task objective:
#
# total_loss =
#     LOCALIZATION_LOSS_WEIGHT × localization_loss
#     + DAMAGE_LOSS_WEIGHT × damage_loss
LOCALIZATION_LOSS_WEIGHT = 1.0
DAMAGE_LOSS_WEIGHT = 1.0


# ------------------------------------------------------------
# Sampling
# ------------------------------------------------------------

USE_WEIGHTED_SAMPLER = True

INTACT_SAMPLE_WEIGHT = 1.0
DAMAGE_SAMPLE_WEIGHT = 5.0
DESTROYED_SAMPLE_WEIGHT = 6.0


# ------------------------------------------------------------
# Model-selection metric
# ------------------------------------------------------------

# Select checkpoint using the two rare and operationally
# important classes: Damage and Destroyed.
MODEL_SELECTION_METRIC = "miou_rare_2class"


# ------------------------------------------------------------
# Reproducibility and device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ------------------------------------------------------------
# Configuration summary
# ------------------------------------------------------------

print("=" * 72)
print("CHANGEOS-R18 CONFIGURATION")
print("=" * 72)
print("Model                  :", MODEL_NAME)
print("Run                    :", RUN_NAME)
print("Device                 :", DEVICE)
print("Paired data            :", PAIRED_DATA_ROOT)
print("Existing split         :", EXPERIMENT_SPLIT_PATH)
print("Output                 :", OUTPUT_DIR)
print("Model directory        :", MODEL_DIR)
print("Results directory      :", RESULTS_DIR)
print("History directory      :", HISTORY_DIR)
print("Best checkpoint        :", BEST_MODEL_PATH)
print("Epochs                 :", EPOCHS)
print("Train batch size       :", TRAIN_BATCH_SIZE)
print("Gradient accumulation  :", GRADIENT_ACCUMULATION_STEPS)
print(
    "Effective batch size   :",
    TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
)
print("Validation batch size  :", VAL_BATCH_SIZE)
print("ImageNet pretrained    :", IMAGENET_PRETRAINED)
print("Damage classes         :", NUM_DAMAGE_CLASSES)
print("Selection metric       :", MODEL_SELECTION_METRIC)
print("=" * 72)

CHANGEOS-R18 CONFIGURATION
Model                  : ChangeOS_R18
Run                    : budget500_seed42
Device                 : cuda
Paired data            : /content/ebd_target_prepost_npz
Existing split         : /content/drive/MyDrive/STRATA_runs/splits/ebd_npz_event_EQTURKEY_TEXASTORNADO_labeled_unlabeled_500.npz
Output                 : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42
Model directory        : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models
Results directory      : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/results
History directory      : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/history
Best checkpoint        : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models/best_changeos_r18.pth
Epochs                 : 100
Train batch size       : 16
Gradient accumulation  : 4
E

In [18]:
# ============================================================
# CHANGEOS-R18:
# SPLIT, LOSS, TRAINING, VALIDATION, CHECKPOINTS, AND EXCEL
# ============================================================

import os
import gc
import random

from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.amp import autocast, GradScaler
from torch.utils.data import WeightedRandomSampler

from tqdm.auto import tqdm


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

def set_experiment_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_experiment_seed(SEED)


# ============================================================
# 2. LOAD EXISTING STRATA SPLIT
# ============================================================

def map_split_paths(
    old_paths,
    new_data_root,
    split_name,
):
    mapped_paths = []
    missing_paths = []

    for old_path in old_paths:
        filename = os.path.basename(
            str(old_path)
        )

        new_path = (
            Path(new_data_root) /
            filename
        )

        if new_path.exists():
            mapped_paths.append(
                str(new_path)
            )

        else:
            missing_paths.append(
                str(new_path)
            )

    if missing_paths:
        print(
            f"Missing {split_name} files: "
            f"{len(missing_paths)}"
        )

        for path in missing_paths[:20]:
            print(path)

        raise FileNotFoundError(
            f"Existing {split_name} split "
            f"could not be mapped completely."
        )

    if len(mapped_paths) != len(set(mapped_paths)):
        raise RuntimeError(
            f"Duplicate files detected in "
            f"{split_name} split."
        )

    return sorted(
        mapped_paths
    )


def load_changeos_split():
    if not EXPERIMENT_SPLIT_PATH.exists():
        raise FileNotFoundError(
            f"Split not found: "
            f"{EXPERIMENT_SPLIT_PATH}"
        )

    with np.load(
        EXPERIMENT_SPLIT_PATH,
        allow_pickle=True,
    ) as split_data:
        required_keys = {
            "labeled",
            "val",
        }

        available_keys = set(
            split_data.files
        )

        if not required_keys.issubset(
            available_keys
        ):
            raise KeyError(
                f"Required keys: {required_keys}; "
                f"available keys: {available_keys}"
            )

        old_train_files = list(
            split_data["labeled"]
        )

        old_val_files = list(
            split_data["val"]
        )

    train_files = map_split_paths(
        old_paths=old_train_files,
        new_data_root=PAIRED_DATA_ROOT,
        split_name="training",
    )

    val_files = map_split_paths(
        old_paths=old_val_files,
        new_data_root=PAIRED_DATA_ROOT,
        split_name="validation",
    )

    overlap = (
        set(train_files)
        & set(val_files)
    )

    if overlap:
        raise RuntimeError(
            f"Train/validation leakage: "
            f"{len(overlap)} files."
        )

    return (
        train_files,
        val_files,
    )


# ============================================================
# 3. SPLIT INSPECTION
# ============================================================

def inspect_split(
    files,
    split_name,
):
    event_counts = Counter()
    class_counts = Counter()

    for file_path in tqdm(
        files,
        desc=f"Inspecting {split_name}",
        leave=False,
    ):
        event_counts[
            get_event_name(file_path)
        ] += 1

        with np.load(file_path) as data:
            labels = np.unique(
                data["damage_mask"]
            )

        for label in labels:
            if int(label) != 0:
                class_counts[
                    int(label)
                ] += 1

    print(
        f"\n===== {split_name.upper()} ====="
    )

    print("Total files:", len(files))
    print("Events:")

    for event_name, count in sorted(
        event_counts.items()
    ):
        print(
            f"  {event_name}: {count:,}"
        )

    print("Files containing classes:")
    print(
        f"  Intact: "
        f"{class_counts[1]:,}"
    )
    print(
        f"  Damage: "
        f"{class_counts[2]:,}"
    )
    print(
        f"  Destroyed: "
        f"{class_counts[3]:,}"
    )


# ============================================================
# 4. WEIGHTED SAMPLER
# ============================================================

def compute_sample_weights(files):
    weights = []

    for file_path in tqdm(
        files,
        desc="Computing sample weights",
        leave=False,
    ):
        with np.load(file_path) as data:
            labels = set(
                np.unique(
                    data["damage_mask"]
                ).tolist()
            )

        sample_weight = (
            INTACT_SAMPLE_WEIGHT
        )

        if 2 in labels:
            sample_weight = max(
                sample_weight,
                DAMAGE_SAMPLE_WEIGHT,
            )

        if 3 in labels:
            sample_weight = max(
                sample_weight,
                DESTROYED_SAMPLE_WEIGHT,
            )

        weights.append(
            sample_weight
        )

    return torch.tensor(
        weights,
        dtype=torch.double,
    )


# ============================================================
# 5. DATALOADERS
# ============================================================

def build_changeos_dataloaders():
    train_files, val_files = (
        load_changeos_split()
    )

    inspect_split(
        train_files,
        "training",
    )

    inspect_split(
        val_files,
        "validation",
    )

    train_dataset = (
        PairedBuildingDamageDataset(
            files=train_files,
            augment=True,
            cache_size=32,
        )
    )

    val_dataset = (
        PairedBuildingDamageDataset(
            files=val_files,
            augment=False,
            cache_size=32,
        )
    )

    if USE_WEIGHTED_SAMPLER:
        sample_weights = (
            compute_sample_weights(
                train_files
            )
        )

        sampler_generator = (
            torch.Generator()
        )

        sampler_generator.manual_seed(
            SEED
        )

        train_sampler = (
            WeightedRandomSampler(
                weights=sample_weights,
                num_samples=len(
                    sample_weights
                ),
                replacement=True,
                generator=sampler_generator,
            )
        )

        train_loader = get_paired_loader(
            dataset=train_dataset,
            batch_size=TRAIN_BATCH_SIZE,
            shuffle=False,
            sampler=train_sampler,
            drop_last=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        )

    else:
        train_loader = get_paired_loader(
            dataset=train_dataset,
            batch_size=TRAIN_BATCH_SIZE,
            shuffle=True,
            sampler=None,
            drop_last=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        )

    val_loader = get_paired_loader(
        dataset=val_dataset,
        batch_size=VAL_BATCH_SIZE,
        shuffle=False,
        sampler=None,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    print("\n===== SPLIT CHECK =====")
    print("Train:", len(train_files))
    print("Validation:", len(val_files))
    print(
        "Overlap:",
        len(
            set(train_files)
            & set(val_files)
        ),
    )

    return (
        train_loader,
        val_loader,
        train_files,
        val_files,
    )


# ============================================================
# 6. CHANGEOS JOINT LOSS
# ============================================================

class ChangeOSJointLoss(nn.Module):
    """
    Joint ChangeOS objective:

        total =
            localization_weight * BCEWithLogits(localization)
            + damage_weight * CrossEntropy(damage)

    localization_logits:
        [B,1,H,W]

    building_target:
        [B,H,W], binary

    damage_logits:
        [B,4,H,W]

    damage_target:
        [B,H,W], values 0..3
    """

    def __init__(
        self,
        localization_pos_weight,
        damage_class_weights,
        localization_loss_weight=1.0,
        damage_loss_weight=1.0,
    ):
        super().__init__()

        self.register_buffer(
            "localization_pos_weight",
            torch.tensor(
                [localization_pos_weight],
                dtype=torch.float32,
            ),
        )

        self.register_buffer(
            "damage_class_weights",
            torch.tensor(
                damage_class_weights,
                dtype=torch.float32,
            ),
        )

        self.localization_loss_weight = float(
            localization_loss_weight
        )

        self.damage_loss_weight = float(
            damage_loss_weight
        )

    def forward(
        self,
        localization_logits,
        damage_logits,
        building_target,
        damage_target,
    ):
        if localization_logits.ndim != 4:
            raise ValueError(
                "localization_logits must be BCHW."
            )

        if localization_logits.shape[1] != 1:
            raise ValueError(
                "ChangeOS localization must have one channel."
            )

        building_target = (
            building_target.float()
        )

        if building_target.ndim == 3:
            building_target = (
                building_target.unsqueeze(1)
            )

        building_target = (
            building_target > 0
        ).float()

        localization_loss = (
            nn.functional.binary_cross_entropy_with_logits(
                localization_logits,
                building_target,
                pos_weight=(
                    self.localization_pos_weight
                ),
            )
        )

        damage_loss = (
            nn.functional.cross_entropy(
                damage_logits,
                damage_target.long(),
                weight=self.damage_class_weights,
            )
        )

        total_loss = (
            self.localization_loss_weight
            * localization_loss
            +
            self.damage_loss_weight
            * damage_loss
        )

        return {
            "total": total_loss,
            "localization": localization_loss,
            "damage": damage_loss,
        }


# ============================================================
# 7. CONFUSION MATRIX
# ============================================================

def update_confusion_matrix(
    confusion_matrix,
    prediction,
    target,
    valid_mask,
    num_classes=4,
):
    prediction = prediction.detach().reshape(-1)
    target = target.detach().reshape(-1)
    valid_mask = valid_mask.detach().reshape(-1).bool()

    valid = (
        valid_mask
        & (target >= 0)
        & (target < num_classes)
        & (prediction >= 0)
        & (prediction < num_classes)
    )

    prediction = prediction[valid]
    target = target[valid]

    if target.numel() == 0:
        return confusion_matrix

    indexes = (
        target * num_classes
        + prediction
    )

    batch_matrix = torch.bincount(
        indexes,
        minlength=num_classes ** 2,
    ).reshape(
        num_classes,
        num_classes,
    )

    return (
        confusion_matrix
        + batch_matrix.cpu()
    )


def update_event_confusion_matrices(
    event_confusion_matrices,
    prediction,
    target,
    valid_mask,
    event_names,
):
    for sample_index, event_name in enumerate(
        event_names
    ):
        event_name = str(
            event_name
        )

        if event_name not in event_confusion_matrices:
            event_confusion_matrices[
                event_name
            ] = torch.zeros(
                NUM_DAMAGE_CLASSES,
                NUM_DAMAGE_CLASSES,
                dtype=torch.long,
            )

        event_confusion_matrices[event_name] = (
            update_confusion_matrix(
                confusion_matrix=(
                    event_confusion_matrices[
                        event_name
                    ]
                ),
                prediction=prediction[
                    sample_index
                ],
                target=target[
                    sample_index
                ],
                valid_mask=valid_mask[
                    sample_index
                ],
                num_classes=NUM_DAMAGE_CLASSES,
            )
        )


# ============================================================
# 8. DAMAGE METRICS
# ============================================================

def compute_damage_metrics(
    confusion_matrix,
):
    """
    Correct three-class building-damage evaluation.

    True foreground classes:
        1 = Intact
        2 = Damage
        3 = Destroyed

    Predicted class 0 is retained and counted as a false
    negative when the ground truth is a building class.
    """
    matrix = confusion_matrix.detach().cpu().numpy().astype(
        np.float64
    )

    foreground_classes = [
        1,
        2,
        3,
    ]

    true_positive = np.array([
        matrix[class_id, class_id]
        for class_id in foreground_classes
    ])

    false_positive = np.array([
        matrix[:, class_id].sum()
        - matrix[class_id, class_id]
        for class_id in foreground_classes
    ])

    false_negative = np.array([
        matrix[class_id, :].sum()
        - matrix[class_id, class_id]
        for class_id in foreground_classes
    ])

    foreground_total = (
        matrix[1:, :].sum()
    )

    accuracy = (
        true_positive.sum()
        / (foreground_total + 1e-6)
    )

    precision = (
        true_positive
        / (
            true_positive
            + false_positive
            + 1e-6
        )
    )

    recall = (
        true_positive
        / (
            true_positive
            + false_negative
            + 1e-6
        )
    )

    f1 = (
        2.0 * precision * recall
        / (
            precision
            + recall
            + 1e-6
        )
    )

    iou = (
        true_positive
        / (
            true_positive
            + false_positive
            + false_negative
            + 1e-6
        )
    )

    kappa_matrix = matrix.copy()
    kappa_matrix[0, :] = 0.0

    total = kappa_matrix.sum()

    observed_agreement = (
        np.trace(kappa_matrix)
        / (total + 1e-6)
    )

    expected_agreement = (
        (
            kappa_matrix.sum(axis=0)
            * kappa_matrix.sum(axis=1)
        ).sum()
        / ((total ** 2) + 1e-6)
    )

    kappa = (
        observed_agreement
        - expected_agreement
    ) / (
        1.0
        - expected_agreement
        + 1e-6
    )

    class_names = [
        "intact",
        "damage",
        "destroyed",
    ]

    metrics = {
        "accuracy": float(accuracy),
        "precision_macro": float(
            np.nanmean(precision)
        ),
        "recall_macro": float(
            np.nanmean(recall)
        ),
        "f1_macro": float(
            np.nanmean(f1)
        ),
        "kappa": float(kappa),
        "miou_3class": float(
            np.nanmean(iou)
        ),
        "miou_rare_2class": float(
            np.nanmean(iou[1:3])
        ),
    }

    for class_index, class_name in enumerate(
        class_names
    ):
        metrics[
            f"precision_{class_name}"
        ] = float(
            precision[class_index]
        )

        metrics[
            f"recall_{class_name}"
        ] = float(
            recall[class_index]
        )

        metrics[
            f"f1_{class_name}"
        ] = float(
            f1[class_index]
        )

        metrics[
            f"iou_{class_name}"
        ] = float(
            iou[class_index]
        )

    return metrics


# ============================================================
# 9. LOCALIZATION METRIC
# ============================================================

def update_localization_intersection_union(
    intersection,
    union,
    localization_logits,
    building_target,
):
    prediction = (
        localization_logits[:, 0] > 0.0
    )

    target = (
        building_target > 0
    )

    intersection += (
        prediction & target
    ).sum().item()

    union += (
        prediction | target
    ).sum().item()

    return (
        intersection,
        union,
    )


# ============================================================
# 10. ONE TRAINING EPOCH
# ============================================================

def train_changeos_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
):
    model.train()

    optimizer.zero_grad(
        set_to_none=True
    )

    accumulated_losses = defaultdict(
        float
    )

    damage_confusion_matrix = torch.zeros(
        NUM_DAMAGE_CLASSES,
        NUM_DAMAGE_CLASSES,
        dtype=torch.long,
    )

    localization_intersection = 0
    localization_union = 0

    number_of_batches = len(
        loader
    )

    progress_bar = tqdm(
        enumerate(loader),
        total=number_of_batches,
        desc="ChangeOS training",
        leave=False,
    )

    for batch_index, batch in progress_bar:
        (
            pre_image,
            post_image,
            damage_target,
            building_target,
            event_names,
        ) = batch[:5]

        pre_image = pre_image.to(
            DEVICE,
            non_blocking=True,
        )

        post_image = post_image.to(
            DEVICE,
            non_blocking=True,
        )

        damage_target = damage_target.to(
            DEVICE,
            non_blocking=True,
        )

        building_target = building_target.to(
            DEVICE,
            non_blocking=True,
        )

        with autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP,
        ):
            (
                localization_logits,
                damage_logits,
            ) = model(
                pre_image,
                post_image,
            )

            loss_dictionary = criterion(
                localization_logits=
                    localization_logits,

                damage_logits=
                    damage_logits,

                building_target=
                    building_target,

                damage_target=
                    damage_target,
            )

            backward_loss = (
                loss_dictionary["total"]
                / GRADIENT_ACCUMULATION_STEPS
            )

        scaler.scale(
            backward_loss
        ).backward()

        is_accumulation_boundary = (
            (batch_index + 1)
            % GRADIENT_ACCUMULATION_STEPS
            == 0
        )

        is_last_batch = (
            batch_index + 1
            == number_of_batches
        )

        if (
            is_accumulation_boundary
            or is_last_batch
        ):
            scaler.unscale_(
                optimizer
            )

            if MAX_GRAD_NORM is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM,
                )

            scaler.step(
                optimizer
            )

            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )

        for loss_name, loss_value in (
            loss_dictionary.items()
        ):
            accumulated_losses[
                loss_name
            ] += float(
                loss_value.detach().item()
            )

        with torch.no_grad():
            damage_prediction = torch.argmax(
                damage_logits,
                dim=1,
            )

            valid_building_mask = (
                building_target > 0
            )

            damage_confusion_matrix = (
                update_confusion_matrix(
                    confusion_matrix=
                        damage_confusion_matrix,

                    prediction=
                        damage_prediction,

                    target=
                        damage_target,

                    valid_mask=
                        valid_building_mask,

                    num_classes=
                        NUM_DAMAGE_CLASSES,
                )
            )

            (
                localization_intersection,
                localization_union,
            ) = update_localization_intersection_union(
                intersection=
                    localization_intersection,

                union=
                    localization_union,

                localization_logits=
                    localization_logits,

                building_target=
                    building_target,
            )

        progress_bar.set_postfix({
            "loss": (
                f"{loss_dictionary['total'].item():.4f}"
            )
        })

    mean_losses = {
        loss_name: loss_sum / max(
            number_of_batches,
            1,
        )
        for loss_name, loss_sum
        in accumulated_losses.items()
    }

    damage_metrics = compute_damage_metrics(
        damage_confusion_matrix
    )

    localization_iou = (
        localization_intersection
        / (
            localization_union
            + 1e-6
        )
    )

    return {
        "losses": mean_losses,
        "localization_iou": float(
            localization_iou
        ),
        "damage_metrics": damage_metrics,
    }


# ============================================================
# 11. VALIDATION: GLOBAL AND EVENT-BASED
# ============================================================

def validate_changeos(
    model,
    loader,
    criterion,
):
    model.eval()

    accumulated_losses = defaultdict(
        float
    )

    global_confusion_matrix = torch.zeros(
        NUM_DAMAGE_CLASSES,
        NUM_DAMAGE_CLASSES,
        dtype=torch.long,
    )

    event_confusion_matrices = {}

    localization_intersection = 0
    localization_union = 0

    event_localization_counts = defaultdict(
        lambda: {
            "intersection": 0,
            "union": 0,
        }
    )

    number_of_batches = len(
        loader
    )

    with torch.inference_mode():
        for batch in tqdm(
            loader,
            desc="ChangeOS validation",
            leave=False,
        ):
            (
                pre_image,
                post_image,
                damage_target,
                building_target,
                event_names,
            ) = batch[:5]

            pre_image = pre_image.to(
                DEVICE,
                non_blocking=True,
            )

            post_image = post_image.to(
                DEVICE,
                non_blocking=True,
            )

            damage_target = damage_target.to(
                DEVICE,
                non_blocking=True,
            )

            building_target = building_target.to(
                DEVICE,
                non_blocking=True,
            )

            with autocast(
                device_type=DEVICE.type,
                enabled=USE_AMP,
            ):
                (
                    localization_logits,
                    damage_logits,
                ) = model(
                    pre_image,
                    post_image,
                )

                loss_dictionary = criterion(
                    localization_logits=
                        localization_logits,

                    damage_logits=
                        damage_logits,

                    building_target=
                        building_target,

                    damage_target=
                        damage_target,
                )

            for loss_name, loss_value in (
                loss_dictionary.items()
            ):
                accumulated_losses[
                    loss_name
                ] += float(
                    loss_value.detach().item()
                )

            damage_prediction = torch.argmax(
                damage_logits,
                dim=1,
            )

            valid_building_mask = (
                building_target > 0
            )

            global_confusion_matrix = (
                update_confusion_matrix(
                    confusion_matrix=
                        global_confusion_matrix,

                    prediction=
                        damage_prediction,

                    target=
                        damage_target,

                    valid_mask=
                        valid_building_mask,

                    num_classes=
                        NUM_DAMAGE_CLASSES,
                )
            )

            update_event_confusion_matrices(
                event_confusion_matrices=
                    event_confusion_matrices,

                prediction=
                    damage_prediction,

                target=
                    damage_target,

                valid_mask=
                    valid_building_mask,

                event_names=
                    event_names,
            )

            (
                localization_intersection,
                localization_union,
            ) = update_localization_intersection_union(
                intersection=
                    localization_intersection,

                union=
                    localization_union,

                localization_logits=
                    localization_logits,

                building_target=
                    building_target,
            )

            localization_prediction = (
                localization_logits[:, 0]
                > 0.0
            )

            localization_target = (
                building_target > 0
            )

            for sample_index, event_name in enumerate(
                event_names
            ):
                event_name = str(
                    event_name
                )

                event_intersection = (
                    localization_prediction[
                        sample_index
                    ]
                    & localization_target[
                        sample_index
                    ]
                ).sum().item()

                event_union = (
                    localization_prediction[
                        sample_index
                    ]
                    | localization_target[
                        sample_index
                    ]
                ).sum().item()

                event_localization_counts[
                    event_name
                ]["intersection"] += (
                    event_intersection
                )

                event_localization_counts[
                    event_name
                ]["union"] += (
                    event_union
                )

    mean_losses = {
        loss_name: loss_sum / max(
            number_of_batches,
            1,
        )
        for loss_name, loss_sum
        in accumulated_losses.items()
    }

    global_damage_metrics = (
        compute_damage_metrics(
            global_confusion_matrix
        )
    )

    event_damage_metrics = {
        event_name: compute_damage_metrics(
            confusion_matrix
        )
        for event_name, confusion_matrix
        in sorted(
            event_confusion_matrices.items()
        )
    }

    global_localization_iou = (
        localization_intersection
        / (
            localization_union
            + 1e-6
        )
    )

    event_localization_iou = {
        event_name: (
            counts["intersection"]
            / (
                counts["union"]
                + 1e-6
            )
        )
        for event_name, counts
        in sorted(
            event_localization_counts.items()
        )
    }

    return {
        "losses": mean_losses,
        "localization_iou": float(
            global_localization_iou
        ),
        "event_localization_iou":
            event_localization_iou,
        "damage_metrics":
            global_damage_metrics,
        "event_damage_metrics":
            event_damage_metrics,
        "global_confusion_matrix":
            global_confusion_matrix,
        "event_confusion_matrices":
            event_confusion_matrices,
    }


# ============================================================
# 12. PRINT VALIDATION RESULTS
# ============================================================

def print_changeos_validation_results(
    results,
    epoch=None,
):
    title = (
        "CHANGEOS-R18 VALIDATION"
        if epoch is None
        else f"CHANGEOS-R18 VALIDATION — EPOCH {epoch}"
    )

    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

    print(
        "Total loss:",
        f"{results['losses']['total']:.4f}",
    )

    print(
        "Localization loss:",
        f"{results['losses']['localization']:.4f}",
    )

    print(
        "Damage loss:",
        f"{results['losses']['damage']:.4f}",
    )

    print(
        "Localization IoU:",
        f"{results['localization_iou']:.4f}",
    )

    metrics = results[
        "damage_metrics"
    ]

    print(
        "Accuracy:",
        f"{metrics['accuracy']:.4f}",
    )

    print(
        "Macro F1:",
        f"{metrics['f1_macro']:.4f}",
    )

    print(
        "Kappa:",
        f"{metrics['kappa']:.4f}",
    )

    print(
        "mIoU-3class:",
        f"{metrics['miou_3class']:.4f}",
    )

    print(
        "mIoU-rare-2class:",
        f"{metrics['miou_rare_2class']:.4f}",
    )

    for class_name in [
        "intact",
        "damage",
        "destroyed",
    ]:
        display_name = (
            class_name.capitalize()
        )

        print(
            f"{display_name}: "
            f"F1={metrics[f'f1_{class_name}']:.4f}, "
            f"IoU={metrics[f'iou_{class_name}']:.4f}"
        )


# ============================================================
# 13. EXCEL OUTPUT
# ============================================================

def metrics_to_row(
    scope,
    event_name,
    localization_iou,
    damage_metrics,
):
    return {
        "scope": scope,
        "event": event_name,
        "model": MODEL_NAME,
        "run": RUN_NAME,
        "budget_per_class":
            TARGET_LABELS_PER_CLASS,
        "localization_iou":
            localization_iou,
        **damage_metrics,
    }


def save_changeos_global_and_event_excel(
    results,
    output_path,
):
    rows = []

    # Global result is deliberately the first row.
    rows.append(
        metrics_to_row(
            scope="global",
            event_name="Global_All_Events",
            localization_iou=(
                results[
                    "localization_iou"
                ]
            ),
            damage_metrics=(
                results[
                    "damage_metrics"
                ]
            ),
        )
    )

    for event_name in sorted(
        results[
            "event_damage_metrics"
        ].keys()
    ):
        rows.append(
            metrics_to_row(
                scope="event",
                event_name=event_name,
                localization_iou=(
                    results[
                        "event_localization_iou"
                    ].get(
                        event_name,
                        np.nan,
                    )
                ),
                damage_metrics=(
                    results[
                        "event_damage_metrics"
                    ][event_name]
                ),
            )
        )

    metrics_dataframe = pd.DataFrame(
        rows
    )

    class_labels = [
        "background",
        "intact",
        "damage",
        "destroyed",
    ]

    global_confusion_dataframe = (
        pd.DataFrame(
            results[
                "global_confusion_matrix"
            ].numpy(),
            index=[
                f"true_{name}"
                for name in class_labels
            ],
            columns=[
                f"pred_{name}"
                for name in class_labels
            ],
        )
    )

    history_dataframe = pd.DataFrame(
        results.get(
            "history",
            []
        )
    )

    with pd.ExcelWriter(
        output_path,
        engine="openpyxl",
    ) as writer:
        metrics_dataframe.to_excel(
            writer,
            sheet_name="Metrics",
            index=False,
        )

        global_confusion_dataframe.to_excel(
            writer,
            sheet_name="Global_CM",
        )

        for event_name, matrix in sorted(
            results[
                "event_confusion_matrices"
            ].items()
        ):
            safe_event_name = (
                event_name
                .replace("-", "_")[:24]
            )

            pd.DataFrame(
                matrix.numpy(),
                index=[
                    f"true_{name}"
                    for name in class_labels
                ],
                columns=[
                    f"pred_{name}"
                    for name in class_labels
                ],
            ).to_excel(
                writer,
                sheet_name=(
                    f"CM_{safe_event_name}"
                )[:31],
            )

        if not history_dataframe.empty:
            history_dataframe.to_excel(
                writer,
                sheet_name="History",
                index=False,
            )

    print(
        "Excel saved:",
        output_path,
    )

    return metrics_dataframe


# ============================================================
# 14. TEXT SUMMARY
# ============================================================

def save_changeos_text_summary(
    results,
    output_path,
):
    with open(
        output_path,
        "w",
        encoding="utf-8",
    ) as output_file:
        output_file.write(
            "ChangeOS-R18 Results\n"
        )

        output_file.write(
            "=" * 60 + "\n"
        )

        output_file.write(
            f"Run: {RUN_NAME}\n"
        )

        output_file.write(
            f"Budget per class: "
            f"{TARGET_LABELS_PER_CLASS}\n"
        )

        output_file.write(
            f"Best checkpoint: "
            f"{BEST_MODEL_PATH}\n\n"
        )

        output_file.write(
            "GLOBAL RESULTS\n"
        )

        output_file.write(
            f"Localization IoU: "
            f"{results['localization_iou']:.6f}\n"
        )

        for metric_name, metric_value in (
            results[
                "damage_metrics"
            ].items()
        ):
            output_file.write(
                f"{metric_name}: "
                f"{metric_value:.6f}\n"
            )

        output_file.write(
            "\nEVENT-BASED RESULTS\n"
        )

        for event_name, metrics in (
            results[
                "event_damage_metrics"
            ].items()
        ):
            output_file.write(
                f"\n[{event_name}]\n"
            )

            output_file.write(
                "localization_iou: "
                f"{results['event_localization_iou'].get(event_name, np.nan):.6f}\n"
            )

            for metric_name, metric_value in (
                metrics.items()
            ):
                output_file.write(
                    f"{metric_name}: "
                    f"{metric_value:.6f}\n"
                )


# ============================================================
# 15. COMPLETE TRAINING PROCEDURE
# ============================================================

def train_changeos_r18():
    set_experiment_seed(
        SEED
    )

    (
        train_loader,
        val_loader,
        train_files,
        val_files,
    ) = build_changeos_dataloaders()

    model = ChangeOSR18(
        damage_classes=NUM_DAMAGE_CLASSES,
        imagenet_pretrained=IMAGENET_PRETRAINED,
        input_is_bgr=INPUT_IS_BGR,
        decoder_channels=(
            CHANGEOS_DECODER_CHANNELS
        ),
    ).to(
        DEVICE
    )

    criterion = ChangeOSJointLoss(
        localization_pos_weight=(
            LOCALIZATION_POS_WEIGHT
        ),
        damage_class_weights=(
            DAMAGE_CLASS_WEIGHTS
        ),
        localization_loss_weight=(
            LOCALIZATION_LOSS_WEIGHT
        ),
        damage_loss_weight=(
            DAMAGE_LOSS_WEIGHT
        ),
    ).to(
        DEVICE
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=SCHEDULER_FACTOR,
            patience=SCHEDULER_PATIENCE,
            min_lr=MIN_LEARNING_RATE,
        )
    )

    scaler = GradScaler(
        device=DEVICE.type,
        enabled=USE_AMP,
    )

    best_score = -float("inf")
    best_epoch = -1
    epochs_without_improvement = 0

    history = []

    print("\n" + "=" * 72)
    print("STARTING CHANGEOS-R18 TRAINING")
    print("=" * 72)
    print("Training samples  :", len(train_files))
    print("Validation samples:", len(val_files))
    print("Device            :", DEVICE)
    print("Best model path   :", BEST_MODEL_PATH)
    print("=" * 72)

    for epoch in range(
        1,
        EPOCHS + 1,
    ):
        current_learning_rate = (
            optimizer.param_groups[0]["lr"]
        )

        print(
            f"\nEpoch {epoch}/{EPOCHS} | "
            f"LR={current_learning_rate:.8f}"
        )

        training_results = (
            train_changeos_one_epoch(
                model=model,
                loader=train_loader,
                criterion=criterion,
                optimizer=optimizer,
                scaler=scaler,
            )
        )

        history_row = {
            "epoch": epoch,
            "learning_rate":
                current_learning_rate,
            "train_total_loss":
                training_results[
                    "losses"
                ]["total"],
            "train_localization_loss":
                training_results[
                    "losses"
                ]["localization"],
            "train_damage_loss":
                training_results[
                    "losses"
                ]["damage"],
            "train_localization_iou":
                training_results[
                    "localization_iou"
                ],
            "train_miou_3class":
                training_results[
                    "damage_metrics"
                ]["miou_3class"],
            "train_miou_rare_2class":
                training_results[
                    "damage_metrics"
                ]["miou_rare_2class"],
        }

        print(
            "Train total loss:",
            f"{training_results['losses']['total']:.4f}",
        )

        should_validate = (
            epoch % EVAL_EVERY == 0
            or epoch == EPOCHS
        )

        if should_validate:
            validation_results = (
                validate_changeos(
                    model=model,
                    loader=val_loader,
                    criterion=criterion,
                )
            )

            print_changeos_validation_results(
                validation_results,
                epoch=epoch,
            )

            validation_metrics = (
                validation_results[
                    "damage_metrics"
                ]
            )

            validation_score = float(
                validation_metrics[
                    MODEL_SELECTION_METRIC
                ]
            )

            scheduler.step(
                validation_score
            )

            history_row.update({
                "val_total_loss":
                    validation_results[
                        "losses"
                    ]["total"],
                "val_localization_loss":
                    validation_results[
                        "losses"
                    ]["localization"],
                "val_damage_loss":
                    validation_results[
                        "losses"
                    ]["damage"],
                "val_localization_iou":
                    validation_results[
                        "localization_iou"
                    ],
                "val_accuracy":
                    validation_metrics[
                        "accuracy"
                    ],
                "val_f1_macro":
                    validation_metrics[
                        "f1_macro"
                    ],
                "val_kappa":
                    validation_metrics[
                        "kappa"
                    ],
                "val_miou_3class":
                    validation_metrics[
                        "miou_3class"
                    ],
                "val_miou_rare_2class":
                    validation_metrics[
                        "miou_rare_2class"
                    ],
            })

            if validation_score > best_score:
                best_score = (
                    validation_score
                )

                best_epoch = epoch

                epochs_without_improvement = 0

                torch.save(
                    {
                        "model":
                            model.state_dict(),
                        "optimizer":
                            optimizer.state_dict(),
                        "scheduler":
                            scheduler.state_dict(),
                        "epoch":
                            epoch,
                        "best_score":
                            best_score,
                        "selection_metric":
                            MODEL_SELECTION_METRIC,
                        "model_name":
                            MODEL_NAME,
                        "run_name":
                            RUN_NAME,
                        "num_damage_classes":
                            NUM_DAMAGE_CLASSES,
                        "decoder_channels":
                            CHANGEOS_DECODER_CHANNELS,
                        "imagenet_pretrained":
                            IMAGENET_PRETRAINED,
                        "input_is_bgr":
                            INPUT_IS_BGR,
                    },
                    BEST_MODEL_PATH,
                )

                print(
                    "Saved best ChangeOS checkpoint: "
                    f"{MODEL_SELECTION_METRIC}="
                    f"{best_score:.4f}"
                )

            else:
                epochs_without_improvement += 1

                print(
                    "No improvement:",
                    f"{epochs_without_improvement}/"
                    f"{EARLY_STOP_PATIENCE}"
                )

        history.append(
            history_row
        )

        torch.save(
            {
                "model":
                    model.state_dict(),
                "optimizer":
                    optimizer.state_dict(),
                "scheduler":
                    scheduler.state_dict(),
                "epoch":
                    epoch,
                "best_score":
                    best_score,
                "best_epoch":
                    best_epoch,
                "history":
                    history,
                "model_name":
                    MODEL_NAME,
                "run_name":
                    RUN_NAME,
            },
            LATEST_MODEL_PATH,
        )

        pd.DataFrame(
            history
        ).to_excel(
            HISTORY_EXCEL_PATH,
            index=False,
        )

        if (
            should_validate
            and epochs_without_improvement
            >= EARLY_STOP_PATIENCE
        ):
            print(
                "\nEarly stopping triggered."
            )
            break

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not BEST_MODEL_PATH.exists():
        raise RuntimeError(
            "Training finished without saving "
            "a best checkpoint."
        )

    print(
        "\nLoading best checkpoint:",
        BEST_MODEL_PATH,
    )

    best_checkpoint = torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(
        best_checkpoint["model"],
        strict=True,
    )

    final_results = validate_changeos(
        model=model,
        loader=val_loader,
        criterion=criterion,
    )

    final_results["history"] = history
    final_results["best_epoch"] = (
        best_checkpoint["epoch"]
    )
    final_results["best_score"] = (
        best_checkpoint["best_score"]
    )

    save_changeos_global_and_event_excel(
        results=final_results,
        output_path=FINAL_EXCEL_PATH,
    )

    save_changeos_text_summary(
        results=final_results,
        output_path=FINAL_TEXT_PATH,
    )

    torch.save(
        {
            "history": history,
            "best_epoch":
                final_results["best_epoch"],
            "best_score":
                final_results["best_score"],
            "selection_metric":
                MODEL_SELECTION_METRIC,
        },
        HISTORY_PATH,
    )

    print("\n" + "=" * 72)
    print("CHANGEOS-R18 TRAINING COMPLETED")
    print("=" * 72)
    print("Best epoch :", final_results["best_epoch"])
    print("Best score :", f"{final_results['best_score']:.4f}")
    print("Checkpoint :", BEST_MODEL_PATH)
    print("Excel      :", FINAL_EXCEL_PATH)
    print("Summary    :", FINAL_TEXT_PATH)
    print("=" * 72)

    return {
        "model": model,
        "results": final_results,
        "history": history,
        "best_checkpoint": str(
            BEST_MODEL_PATH
        ),
        "output_dir": str(
            OUTPUT_DIR
        ),
    }

## Main and Training Step

In [19]:
# ============================================================
# PRE/POST BASELINE WORKSPACE
# ============================================================

from pathlib import Path
from datetime import datetime


# Main workspace containing all bi-temporal baselines
PREPOST_WORKSPACE_ROOT = Path(
    "/content/drive/MyDrive/STRATA_runs/"
    "PrePost_Baselines"
)


# Four selected models
PREPOST_MODEL_NAMES = [
    "Siamese_UNet",
    "ChangeOS_R18",
    "ChangeFormerV6",
    "MambaBDA_Tiny",
]


# Current experiment
CURRENT_MODEL_NAME = "ChangeOS_R18"

TARGET_LABELS_PER_CLASS = 500
SEED = 42

CURRENT_RUN_NAME = (
    f"budget{TARGET_LABELS_PER_CLASS}_"
    f"seed{SEED}"
)


# Create top-level folders for all four models
PREPOST_WORKSPACE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

for model_name in PREPOST_MODEL_NAMES:

    model_root = (
        PREPOST_WORKSPACE_ROOT /
        model_name
    )

    model_root.mkdir(
        parents=True,
        exist_ok=True,
    )


print(
    "Pre/Post workspace:",
    PREPOST_WORKSPACE_ROOT,
)

print("\nModel folders:")

for model_name in PREPOST_MODEL_NAMES:
    print(
        " ",
        PREPOST_WORKSPACE_ROOT /
        model_name
    )


# ============================================================
# GENERIC PRE/POST EXPERIMENT MANAGER
# ============================================================

import os
import sys
import json

import numpy as np

from pathlib import Path
from datetime import datetime
from scipy import io


# ============================================================
# JSON CONVERSION
# ============================================================

def make_json_serializable(value):

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
        ),
    ):
        return value.item()

    if isinstance(value, dict):

        return {
            str(key): make_json_serializable(
                item
            )
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):

        return [
            make_json_serializable(item)
            for item in value
        ]

    return value


# ============================================================
# PREPARE MODEL-SPECIFIC WORKSPACE
# ============================================================

def configure_model_workspace(
    model_name,
    run_name=None,
):
    """
    Create a separate workspace for one pre/post model.

    The global output variables are updated because the
    previously defined training functions use these paths.
    """

    global CURRENT_MODEL_NAME
    global CURRENT_RUN_NAME

    global OUTPUT_DIR

    global MODEL_DIR
    global LOCALIZATION_MODEL_DIR
    global DAMAGE_MODEL_DIR

    global RESULTS_DIR
    global HISTORY_DIR

    global BEST_LOCALIZATION_PATH
    global BEST_DAMAGE_PATH
    global LATEST_LOCALIZATION_PATH
    global LATEST_DAMAGE_PATH

    global FINAL_EXCEL_PATH
    global FINAL_TEXT_PATH
    global FINAL_MAT_PATH
    global FINAL_JSON_PATH
    global HISTORY_PATH

    if model_name not in (
        PREPOST_MODEL_NAMES
    ):
        raise ValueError(
            f"Unknown model: {model_name}. "
            f"Available models: "
            f"{PREPOST_MODEL_NAMES}"
        )

    CURRENT_MODEL_NAME = model_name

    if run_name is None:

        run_name = (
            f"budget"
            f"{TARGET_LABELS_PER_CLASS}_"
            f"seed{SEED}"
        )

    CURRENT_RUN_NAME = run_name

    # --------------------------------------------------------
    # Main run directory
    # --------------------------------------------------------

    OUTPUT_DIR = (
        PREPOST_WORKSPACE_ROOT /
        model_name /
        run_name
    )

    # --------------------------------------------------------
    # Models
    # --------------------------------------------------------

    MODEL_DIR = (
        OUTPUT_DIR /
        "models"
    )

    LOCALIZATION_MODEL_DIR = (
        MODEL_DIR /
        "localization"
    )

    DAMAGE_MODEL_DIR = (
        MODEL_DIR /
        "damage"
    )

    BEST_LOCALIZATION_PATH = (
        LOCALIZATION_MODEL_DIR /
        "best_localization_model.pth"
    )

    BEST_DAMAGE_PATH = (
        DAMAGE_MODEL_DIR /
        "best_damage_model.pth"
    )

    LATEST_LOCALIZATION_PATH = (
        LOCALIZATION_MODEL_DIR /
        "latest_localization_model.pth"
    )

    LATEST_DAMAGE_PATH = (
        DAMAGE_MODEL_DIR /
        "latest_damage_model.pth"
    )

    # --------------------------------------------------------
    # Results and history
    # --------------------------------------------------------

    RESULTS_DIR = (
        OUTPUT_DIR /
        "results"
    )

    HISTORY_DIR = (
        OUTPUT_DIR /
        "history"
    )

    FINAL_EXCEL_PATH = (
        RESULTS_DIR /
        "final_validation_global_and_event_metrics.xlsx"
    )

    FINAL_TEXT_PATH = (
        RESULTS_DIR /
        "final_results.txt"
    )

    FINAL_MAT_PATH = (
        RESULTS_DIR /
        "final_results.mat"
    )

    FINAL_JSON_PATH = (
        RESULTS_DIR /
        "final_results.json"
    )

    HISTORY_PATH = (
        HISTORY_DIR /
        "training_history.pth"
    )

    # --------------------------------------------------------
    # Create directories
    # --------------------------------------------------------

    for directory in [
        OUTPUT_DIR,
        MODEL_DIR,
        LOCALIZATION_MODEL_DIR,
        DAMAGE_MODEL_DIR,
        RESULTS_DIR,
        HISTORY_DIR,
    ]:
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )

    workspace = {
        "model_name": model_name,
        "run_name": run_name,
        "output_dir": OUTPUT_DIR,
        "model_dir": MODEL_DIR,
        "localization_model_dir": (
            LOCALIZATION_MODEL_DIR
        ),
        "damage_model_dir": (
            DAMAGE_MODEL_DIR
        ),
        "results_dir": RESULTS_DIR,
        "history_dir": HISTORY_DIR,
    }

    print("=" * 72)
    print("PRE/POST MODEL WORKSPACE")
    print("=" * 72)

    print(
        f"Model:       {model_name}"
    )

    print(
        f"Run:         {run_name}"
    )

    print(
        f"Budget:      "
        f"{TARGET_LABELS_PER_CLASS} "
        f"samples per class"
    )

    print(
        f"Seed:        {SEED}"
    )

    print(
        f"Output:      {OUTPUT_DIR}"
    )

    print(
        f"Models:      {MODEL_DIR}"
    )

    print(
        f"Results:     {RESULTS_DIR}"
    )

    print(
        f"History:     {HISTORY_DIR}"
    )

    print("=" * 72)

    return workspace


# ============================================================
# SAVE TEXT RESULTS
# ============================================================

def save_results_text(
    experiment_results,
    elapsed_seconds,
):

    final_results = (
        experiment_results[
            "final_results"
        ]
    )

    global_metrics = (
        final_results[
            "global_metrics"
        ]
    )

    event_metrics = (
        final_results[
            "event_metrics"
        ]
    )

    with open(
        FINAL_TEXT_PATH,
        mode="w",
        encoding="utf-8",
    ) as text_file:

        text_file.write(
            "Pre/Post Building Damage "
            "Assessment Results\n"
        )

        text_file.write(
            "=" * 60 + "\n"
        )

        text_file.write(
            f"Model: "
            f"{CURRENT_MODEL_NAME}\n"
        )

        text_file.write(
            f"Run: "
            f"{CURRENT_RUN_NAME}\n"
        )

        text_file.write(
            f"Target budget per class: "
            f"{TARGET_LABELS_PER_CLASS}\n"
        )

        text_file.write(
            f"Seed: {SEED}\n"
        )

        text_file.write(
            f"Runtime seconds: "
            f"{elapsed_seconds:.2f}\n"
        )

        text_file.write(
            f"Output directory: "
            f"{OUTPUT_DIR}\n\n"
        )

        text_file.write(
            "GLOBAL RESULTS\n"
        )

        text_file.write(
            "-" * 60 + "\n"
        )

        for metric_name in [
            "accuracy",
            "precision_macro",
            "recall_macro",
            "f1_macro",
            "kappa",
            "miou_3class",
            "miou_rare_2class",
        ]:

            text_file.write(
                f"{metric_name}: "
                f"{global_metrics[metric_name]}\n"
            )

        class_names = [
            "intact",
            "damage",
            "destroyed",
        ]

        text_file.write(
            "\nGLOBAL CLASS-WISE RESULTS\n"
        )

        text_file.write(
            "-" * 60 + "\n"
        )

        for index, class_name in enumerate(
            class_names
        ):

            text_file.write(
                f"{class_name}_precision: "
                f"{global_metrics['precision_per_class'][index]}\n"
            )

            text_file.write(
                f"{class_name}_recall: "
                f"{global_metrics['recall_per_class'][index]}\n"
            )

            text_file.write(
                f"{class_name}_f1: "
                f"{global_metrics['f1_per_class'][index]}\n"
            )

            text_file.write(
                f"{class_name}_iou: "
                f"{global_metrics['iou_per_class'][index]}\n"
            )

        text_file.write(
            "\nEVENT-LEVEL RESULTS\n"
        )

        text_file.write(
            "=" * 60 + "\n"
        )

        for event_name, metrics in (
            event_metrics.items()
        ):

            text_file.write(
                f"\n{event_name}\n"
            )

            text_file.write(
                "-" * 60 + "\n"
            )

            for metric_name in [
                "accuracy",
                "precision_macro",
                "recall_macro",
                "f1_macro",
                "kappa",
                "miou_3class",
                "miou_rare_2class",
            ]:

                text_file.write(
                    f"{metric_name}: "
                    f"{metrics[metric_name]}\n"
                )

    print(
        f"Text results saved: "
        f"{FINAL_TEXT_PATH}"
    )


# ============================================================
# SAVE MATLAB RESULTS
# ============================================================

def save_results_mat(
    experiment_results,
):

    final_results = (
        experiment_results[
            "final_results"
        ]
    )

    global_metrics = (
        final_results[
            "global_metrics"
        ]
    )

    mat_results = {
        "model_name": (
            CURRENT_MODEL_NAME
        ),
        "run_name": CURRENT_RUN_NAME,
        "target_budget_per_class": (
            TARGET_LABELS_PER_CLASS
        ),
        "seed": SEED,
        "global_accuracy": (
            global_metrics["accuracy"]
        ),
        "global_precision_macro": (
            global_metrics[
                "precision_macro"
            ]
        ),
        "global_recall_macro": (
            global_metrics[
                "recall_macro"
            ]
        ),
        "global_f1_macro": (
            global_metrics["f1_macro"]
        ),
        "global_kappa": (
            global_metrics["kappa"]
        ),
        "global_miou_3class": (
            global_metrics[
                "miou_3class"
            ]
        ),
        "global_miou_rare_2class": (
            global_metrics[
                "miou_rare_2class"
            ]
        ),
        "global_precision_per_class": (
            global_metrics[
                "precision_per_class"
            ]
        ),
        "global_recall_per_class": (
            global_metrics[
                "recall_per_class"
            ]
        ),
        "global_f1_per_class": (
            global_metrics[
                "f1_per_class"
            ]
        ),
        "global_iou_per_class": (
            global_metrics[
                "iou_per_class"
            ]
        ),
        "global_confusion_matrix": (
            final_results[
                "global_confusion_matrix"
            ].numpy()
        ),
    }

    for event_name, metrics in (
        final_results[
            "event_metrics"
        ].items()
    ):

        safe_event_name = (
            event_name
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
        )

        mat_results[
            f"{safe_event_name}_f1_macro"
        ] = metrics["f1_macro"]

        mat_results[
            f"{safe_event_name}_kappa"
        ] = metrics["kappa"]

        mat_results[
            f"{safe_event_name}_miou_3class"
        ] = metrics[
            "miou_3class"
        ]

        mat_results[
            f"{safe_event_name}_miou_rare_2class"
        ] = metrics[
            "miou_rare_2class"
        ]

        mat_results[
            f"{safe_event_name}_iou_per_class"
        ] = metrics[
            "iou_per_class"
        ]

    io.savemat(
        FINAL_MAT_PATH,
        mat_results,
    )

    print(
        f"MAT results saved: "
        f"{FINAL_MAT_PATH}"
    )


# ============================================================
# SAVE JSON RESULTS
# ============================================================

def save_results_json(
    experiment_results,
    elapsed_seconds,
):

    final_results = (
        experiment_results[
            "final_results"
        ]
    )

    json_results = {
        "model": CURRENT_MODEL_NAME,
        "run": CURRENT_RUN_NAME,
        "target_budget_per_class": (
            TARGET_LABELS_PER_CLASS
        ),
        "seed": SEED,
        "runtime_seconds": (
            elapsed_seconds
        ),
        "best_localization_epoch": (
            experiment_results.get(
                "best_localization_epoch"
            )
        ),
        "best_damage_epoch": (
            experiment_results.get(
                "best_damage_epoch"
            )
        ),
        "global_metrics": (
            final_results[
                "global_metrics"
            ]
        ),
        "event_metrics": (
            final_results[
                "event_metrics"
            ]
        ),
    }

    json_results = (
        make_json_serializable(
            json_results
        )
    )

    with open(
        FINAL_JSON_PATH,
        mode="w",
        encoding="utf-8",
    ) as json_file:

        json.dump(
            json_results,
            json_file,
            indent=2,
            ensure_ascii=False,
        )

    print(
        f"JSON results saved: "
        f"{FINAL_JSON_PATH}"
    )


# ============================================================
# GENERIC EXPERIMENT WRAPPER
# ============================================================

def run_prepost_experiment(
    model_name,
    trainer_function,
    run_name=None,
):
    """
    Configure the output workspace, call the selected trainer,
    and save TXT, MAT, JSON, Excel, checkpoints, and history.

    The trainer itself remains model-specific.
    """

    workspace = configure_model_workspace(
        model_name=model_name,
        run_name=run_name,
    )

    start_time = datetime.now()

    print(
        f"\nStarting {model_name}..."
    )

    sys.stdout.flush()

    experiment_results = (
        trainer_function()
    )

    end_time = datetime.now()

    elapsed_seconds = (
        end_time -
        start_time
    ).total_seconds()

    if experiment_results is None:
        raise RuntimeError(
            f"{model_name} trainer returned None."
        )

    if "final_results" not in (
        experiment_results
    ):
        raise KeyError(
            "Trainer output must contain "
            "'final_results'."
        )

    # Excel is created inside the model-specific trainer.
    # The generic wrapper creates additional portable results.
    save_results_text(
        experiment_results,
        elapsed_seconds,
    )

    save_results_mat(
        experiment_results
    )

    save_results_json(
        experiment_results,
        elapsed_seconds,
    )

    print("\n" + "=" * 72)
    print("EXPERIMENT FINISHED")
    print("=" * 72)

    print(
        f"Model:    {model_name}"
    )

    print(
        f"Runtime:  "
        f"{elapsed_seconds / 3600:.2f} hours"
    )

    print(
        f"Models:   {MODEL_DIR}"
    )

    print(
        f"Results:  {RESULTS_DIR}"
    )

    print(
        f"History:  {HISTORY_DIR}"
    )

    print("=" * 72)

    return experiment_results

Pre/Post workspace: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines

Model folders:
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeFormerV6
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/MambaBDA_Tiny


In [22]:
# ============================================================
# VERIFY CHANGEOS-R18 WORKSPACE
# ============================================================

from pathlib import Path


required_output_paths = [
    OUTPUT_DIR,
    MODEL_DIR,
    RESULTS_DIR,
    HISTORY_DIR,
]

print("=" * 72)
print("CHANGEOS-R18 WORKSPACE")
print("=" * 72)
print("Model :", MODEL_NAME)
print("Run   :", RUN_NAME)
print("Output:", OUTPUT_DIR)
print("=" * 72)

for required_path in required_output_paths:
    required_path = Path(
        required_path
    )

    required_path.mkdir(
        parents=True,
        exist_ok=True,
    )

print("\nVerified output directories:")

for required_path in required_output_paths:
    print(" ", required_path)


required_output_files = {
    "Best model":
        BEST_MODEL_PATH,

    "Latest model":
        LATEST_MODEL_PATH,

    "Final Excel":
        FINAL_EXCEL_PATH,

    "Text summary":
        FINAL_TEXT_PATH,

    "History PTH":
        HISTORY_PATH,

    "History Excel":
        HISTORY_EXCEL_PATH,
}

print("\nConfigured output files:")

for file_label, file_path in (
    required_output_files.items()
):
    print(
        f"  {file_label}: {file_path}"
    )


expected_output_dir = Path(
    "/content/drive/MyDrive/STRATA_runs/"
    "PrePost_Baselines/"
    "ChangeOS_R18/"
    "budget500_seed42"
)

if Path(OUTPUT_DIR) != expected_output_dir:
    raise RuntimeError(
        "Unexpected ChangeOS output directory:\n"
        f"Current : {OUTPUT_DIR}\n"
        f"Expected: {expected_output_dir}"
    )

print("\n✅ ChangeOS-R18 workspace is correctly configured.")
print("No training has started.")

CHANGEOS-R18 WORKSPACE
Model : ChangeOS_R18
Run   : budget500_seed42
Output: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42

Verified output directories:
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/results
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/history

Configured output files:
  Best model: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models/best_changeos_r18.pth
  Latest model: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models/latest_changeos_r18.pth
  Final Excel: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/results/final_validation_global_and_event_metrics.xlsx
  Text summa

## Run

In [23]:
# ============================================================
# RUN CHANGEOS-R18 EXPERIMENT
# ============================================================

changeos_experiment = train_changeos_r18()

changeos_results = (
    changeos_experiment["results"]
)

changeos_model = (
    changeos_experiment["model"]
)

print("\n" + "=" * 72)
print("CHANGEOS-R18 EXPERIMENT FINISHED")
print("=" * 72)
print(
    "Best checkpoint:",
    changeos_experiment[
        "best_checkpoint"
    ],
)
print(
    "Output directory:",
    changeos_experiment[
        "output_dir"
    ],
)
print("=" * 72)

Inspecting training:   0%|          | 0/1380 [00:00<?, ?it/s]


===== TRAINING =====
Total files: 1380
Events:
  EARTHQUAKE-TURKEY: 253
  HURRICANE-DELTA: 396
  HURRICANE-IDA: 653
  TEXAS-TORNADOES: 78
Files containing classes:
  Intact: 1,190
  Damage: 567
  Destroyed: 529


Inspecting validation:   0%|          | 0/3931 [00:00<?, ?it/s]


===== VALIDATION =====
Total files: 3931
Events:
  EARTHQUAKE-TURKEY: 771
  HURRICANE-DELTA: 784
  HURRICANE-IDA: 2,191
  TEXAS-TORNADOES: 185
Files containing classes:
  Intact: 3,812
  Damage: 363
  Destroyed: 293


Computing sample weights:   0%|          | 0/1380 [00:00<?, ?it/s]

...

Train total loss: 0.6149


ChangeOS validation:   0%|          | 0/246 [00:00<?, ?it/s]


CHANGEOS-R18 VALIDATION — EPOCH 63
Total loss: 0.7784
Localization loss: 0.3547
Damage loss: 0.4237
Localization IoU: 0.5488
Accuracy: 0.8876
Macro F1: 0.5120
Kappa: 0.2811
mIoU-3class: 0.4131
mIoU-rare-2class: 0.1744
Intact: F1=0.9420, IoU=0.8904
Damage: F1=0.3062, IoU=0.1808
Destroyed: F1=0.2877, IoU=0.1680
No improvement: 5/5

Early stopping triggered.

Loading best checkpoint: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models/best_changeos_r18.pth


ChangeOS validation:   0%|          | 0/246 [00:00<?, ?it/s]

Excel saved: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/results/final_validation_global_and_event_metrics.xlsx

CHANGEOS-R18 TRAINING COMPLETED
Best epoch : 48
Best score : 0.1816
Checkpoint : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models/best_changeos_r18.pth
Excel      : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/results/final_validation_global_and_event_metrics.xlsx
Summary    : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/results/final_results.txt

CHANGEOS-R18 EXPERIMENT FINISHED
Best checkpoint: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models/best_changeos_r18.pth
Output directory: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42


## Predection on independent test event

In [27]:
# ============================================================
# FIX — FETCH ALL FIGSHARE FILES WITH PAGINATION
# ============================================================

import requests


def get_figshare_files(
    article_id,
    page_size=100,
):
    all_files = []
    page = 1

    while True:
        url = (
            f"https://api.figshare.com/v2/articles/"
            f"{article_id}/files"
        )

        response = requests.get(
            url,
            params={
                "page": page,
                "page_size": page_size,
            },
            timeout=120,
        )

        response.raise_for_status()

        page_files = response.json()

        if not isinstance(page_files, list):
            raise RuntimeError(
                f"Invalid Figshare response on page {page}."
            )

        if len(page_files) == 0:
            break

        all_files.extend(page_files)

        print(
            f"Page {page}: "
            f"{len(page_files)} files received"
        )

        if len(page_files) < page_size:
            break

        page += 1

    if len(all_files) == 0:
        raise RuntimeError(
            "Figshare returned no files."
        )

    # حذف موارد تکراری احتمالی براساس file ID
    unique_files = {}

    for file_info in all_files:
        file_id = file_info.get(
            "id",
            file_info.get("name"),
        )

        unique_files[file_id] = file_info

    all_files = list(unique_files.values())

    print("\nTotal unique Figshare files:", len(all_files))

    return all_files

In [29]:
from pathlib import Path

ARTICLE_ID = 25285009
EVENT_NAME = "HURRICANE-DORIAN"

LOCAL_ROOT = Path(
    f"/content/ebd_{EVENT_NAME}_local"
)

DOWNLOAD_DIR = (
    LOCAL_ROOT /
    "downloads"
)

EXTRACT_DIR = (
    LOCAL_ROOT /
    "extracted"
)

DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("ARTICLE_ID  :", ARTICLE_ID)
print("EVENT_NAME  :", EVENT_NAME)
print("DOWNLOAD_DIR:", DOWNLOAD_DIR)

ARTICLE_ID  : 25285009
EVENT_NAME  : HURRICANE-DORIAN
DOWNLOAD_DIR: /content/ebd_HURRICANE-DORIAN_local/downloads


In [30]:
# ============================================================
# FIND DORIAN AFTER COMPLETE PAGINATION
# ============================================================

figshare_files = get_figshare_files(
    ARTICLE_ID,
    page_size=100,
)

print("\nFiles containing DORIAN:")

dorian_candidates = [
    file_info
    for file_info in figshare_files
    if "DORIAN" in file_info["name"].upper()
]

for file_info in dorian_candidates:
    print(
        " ",
        file_info["name"],
        "|",
        f"{file_info.get('size', 0) / 1024**3:.3f} GB",
    )

if len(dorian_candidates) == 0:
    raise FileNotFoundError(
        "No file containing DORIAN was found after "
        "fetching every Figshare API page."
    )

selected_file = find_event_zip(
    files=figshare_files,
    event_name=EVENT_NAME,
)

remote_filename = selected_file["name"]
download_url = selected_file["download_url"]

ZIP_PATH = DOWNLOAD_DIR / remote_filename

print("\nSelected file:")
print("Filename:", remote_filename)
print("URL     :", download_url)
print(
    "Size    :",
    f"{selected_file.get('size', 0) / 1024**3:.3f} GB",
)

Page 1: 12 files received

Total unique Figshare files: 12

Files containing DORIAN:
  HURRICANE-DORIAN.zip | 0.413 GB

Selected file:
Filename: HURRICANE-DORIAN.zip
URL     : https://ndownloader.figshare.com/files/44708263
Size    : 0.413 GB


In [32]:
# ============================================================
# COMPLETE DORIAN DOWNLOAD CELL — SELF-CONTAINED
# ============================================================

import os
import zipfile
import requests

from pathlib import Path
from tqdm.auto import tqdm


# ============================================================
# 1. SETTINGS
# ============================================================

ARTICLE_ID = 25285009
EVENT_NAME = "HURRICANE-DORIAN"

LOCAL_ROOT = Path(
    f"/content/ebd_{EVENT_NAME}_local"
)

DOWNLOAD_DIR = (
    LOCAL_ROOT /
    "downloads"
)

EXTRACT_DIR = (
    LOCAL_ROOT /
    "extracted"
)

DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. GET ALL FIGSHARE FILES
# ============================================================

def get_figshare_files(
    article_id,
    page_size=100,
):
    all_files = []
    page = 1

    while True:
        api_url = (
            f"https://api.figshare.com/v2/articles/"
            f"{article_id}/files"
        )

        response = requests.get(
            api_url,
            params={
                "page": page,
                "page_size": page_size,
            },
            timeout=120,
        )

        response.raise_for_status()

        page_files = response.json()

        if not isinstance(page_files, list):
            raise RuntimeError(
                f"Invalid Figshare response on page {page}."
            )

        if len(page_files) == 0:
            break

        all_files.extend(
            page_files
        )

        print(
            f"Figshare page {page}: "
            f"{len(page_files)} files"
        )

        if len(page_files) < page_size:
            break

        page += 1

    if len(all_files) == 0:
        raise RuntimeError(
            "No files were returned by Figshare."
        )

    unique_files = {}

    for file_information in all_files:
        file_identifier = file_information.get(
            "id",
            file_information["name"],
        )

        unique_files[
            file_identifier
        ] = file_information

    all_files = list(
        unique_files.values()
    )

    print(
        "Total unique Figshare files:",
        len(all_files),
    )

    return all_files


# ============================================================
# 3. FIND DORIAN ZIP
# ============================================================

def find_event_zip(
    files,
    event_name,
):
    event_upper = (
        event_name.upper()
    )

    candidates = [
        file_information
        for file_information in files
        if (
            "DORIAN"
            in file_information["name"].upper()
            and file_information["name"]
            .lower()
            .endswith(".zip")
        )
    ]

    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No Dorian ZIP was found in "
            f"Figshare article {ARTICLE_ID}."
        )

    candidates = sorted(
        candidates,
        key=lambda item: (
            event_upper
            not in item["name"].upper(),
            item["name"],
        ),
    )

    print("\nDorian candidates:")

    for candidate in candidates:
        print(
            " ",
            candidate["name"],
            "|",
            f"{candidate.get('size', 0) / 1024**3:.3f} GB",
        )

    selected = candidates[0]

    print("\nSelected file:")
    print("Name:", selected["name"])
    print("URL :", selected["download_url"])
    print(
        "Size:",
        f"{selected.get('size', 0) / 1024**3:.3f} GB",
    )

    return selected


# ============================================================
# 4. DOWNLOAD FUNCTION
# ============================================================

def download_file(
    download_url,
    destination_path,
    chunk_size=4 * 1024 * 1024,
):
    destination_path = Path(
        destination_path
    )

    partial_path = (
        destination_path.with_suffix(
            destination_path.suffix + ".part"
        )
    )

    if (
        destination_path.exists()
        and destination_path.stat().st_size > 0
    ):
        print("\nExisting download found:")
        print(destination_path)
        print(
            "Size:",
            f"{destination_path.stat().st_size / 1024**3:.3f} GB",
        )

        return destination_path

    # یک فایل ناقص قبلی نباید با دانلود جدید مخلوط شود.
    if partial_path.exists():
        print(
            "Removing incomplete partial file:",
            partial_path,
        )

        partial_path.unlink()

    print("\nDownloading:")
    print(download_url)

    with requests.get(
        download_url,
        stream=True,
        timeout=(60, 300),
    ) as response:
        response.raise_for_status()

        total_size = int(
            response.headers.get(
                "content-length",
                0,
            )
        )

        with open(
            partial_path,
            "wb",
        ) as output_file:
            with tqdm(
                total=(
                    total_size
                    if total_size > 0
                    else None
                ),
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc="Downloading HURRICANE-DORIAN",
            ) as progress_bar:
                for chunk in response.iter_content(
                    chunk_size=chunk_size
                ):
                    if chunk:
                        output_file.write(
                            chunk
                        )

                        progress_bar.update(
                            len(chunk)
                        )

    if not partial_path.exists():
        raise RuntimeError(
            "Download did not create the expected file."
        )

    if partial_path.stat().st_size == 0:
        raise RuntimeError(
            "Downloaded file is empty."
        )

    os.replace(
        partial_path,
        destination_path,
    )

    print("\nDownload completed:")
    print(destination_path)
    print(
        "Downloaded size:",
        f"{destination_path.stat().st_size / 1024**3:.3f} GB",
    )

    return destination_path


# ============================================================
# 5. ZIP VALIDATION
# ============================================================

def validate_zip(
    zip_path,
):
    zip_path = Path(
        zip_path
    )

    print("\nChecking ZIP integrity:")
    print(zip_path)

    if not zip_path.is_file():
        raise FileNotFoundError(
            f"ZIP does not exist: {zip_path}"
        )

    if not zipfile.is_zipfile(
        zip_path
    ):
        raise zipfile.BadZipFile(
            f"Invalid ZIP file: {zip_path}"
        )

    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as archive:
        members = archive.infolist()

        normalized_names = [
            member.filename.replace(
                "\\",
                "/",
            )
            for member in members
        ]

        pre_images = [
            name
            for name in normalized_names
            if (
                "/images/" in name
                and name.endswith(
                    "_pre_disaster.png"
                )
            )
        ]

        post_images = [
            name
            for name in normalized_names
            if (
                "/images/" in name
                and name.endswith(
                    "_post_disaster.png"
                )
            )
        ]

        pre_masks = [
            name
            for name in normalized_names
            if (
                "/masks/" in name
                and name.endswith(
                    "_pre_disaster.png"
                )
            )
        ]

        post_masks = [
            name
            for name in normalized_names
            if (
                "/masks/" in name
                and name.endswith(
                    "_post_disaster.png"
                )
            )
        ]

    report = {
        "archive_members":
            len(members),

        "pre_images":
            len(pre_images),

        "post_images":
            len(post_images),

        "pre_masks":
            len(pre_masks),

        "post_masks":
            len(post_masks),
    }

    print("Archive members:", report["archive_members"])
    print("Pre images     :", report["pre_images"])
    print("Post images    :", report["post_images"])
    print("Pre masks      :", report["pre_masks"])
    print("Post masks     :", report["post_masks"])

    expected_count = 540

    count_values = [
        report["pre_images"],
        report["post_images"],
        report["pre_masks"],
        report["post_masks"],
    ]

    if len(set(count_values)) != 1:
        raise RuntimeError(
            "Pre/post image and mask counts are unequal."
        )

    if count_values[0] != expected_count:
        raise RuntimeError(
            f"Expected {expected_count} paired scenes, "
            f"but found {count_values[0]}."
        )

    print("\n✅ ZIP integrity and structure are valid.")

    return report


# ============================================================
# 6. RUN DOWNLOAD
# ============================================================

print("=" * 72)
print("HURRICANE-DORIAN DOWNLOAD")
print("=" * 72)
print("Article ID :", ARTICLE_ID)
print("Event      :", EVENT_NAME)
print("Local root :", LOCAL_ROOT)
print("=" * 72)

figshare_files = get_figshare_files(
    article_id=ARTICLE_ID,
    page_size=100,
)

selected_file = find_event_zip(
    files=figshare_files,
    event_name=EVENT_NAME,
)

remote_filename = (
    selected_file["name"]
)

download_url = (
    selected_file["download_url"]
)

ZIP_PATH = (
    DOWNLOAD_DIR /
    remote_filename
)

ZIP_PATH = download_file(
    download_url=download_url,
    destination_path=ZIP_PATH,
)

zip_report = validate_zip(
    ZIP_PATH
)

print("\n" + "=" * 72)
print("DORIAN DOWNLOAD COMPLETED SUCCESSFULLY")
print("=" * 72)
print("ZIP path:", ZIP_PATH)
print("Report  :", zip_report)
print("=" * 72)

HURRICANE-DORIAN DOWNLOAD
Article ID : 25285009
Event      : HURRICANE-DORIAN
Local root : /content/ebd_HURRICANE-DORIAN_local
Figshare page 1: 12 files
Total unique Figshare files: 12

Dorian candidates:
  HURRICANE-DORIAN.zip | 0.413 GB

Selected file:
Name: HURRICANE-DORIAN.zip
URL : https://ndownloader.figshare.com/files/44708263
Size: 0.413 GB

Downloading:
https://ndownloader.figshare.com/files/44708263



Download completed:
/content/ebd_HURRICANE-DORIAN_local/downloads/HURRICANE-DORIAN.zip
Downloaded size: 0.413 GB

Checking ZIP integrity:
/content/ebd_HURRICANE-DORIAN_local/downloads/HURRICANE-DORIAN.zip
Archive members: 2163
Pre images     : 540
Post images    : 540
Pre masks      : 540
Post masks     : 540

✅ ZIP integrity and structure are valid.

DORIAN DOWNLOAD COMPLETED SUCCESSFULLY
ZIP path: /content/ebd_HURRICANE-DORIAN_local/downloads/HURRICANE-DORIAN.zip
Report  : {'archive_members': 2163, 'pre_images': 540, 'post_images': 540, 'pre_masks': 540, 'post_masks': 540}


In [34]:
# ============================================================
# STEP 2 — EXTRACT AND VERIFY PRE/POST PAIRING
# ============================================================

import os
import zipfile

from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

EVENT_NAME = "HURRICANE-DORIAN"

LOCAL_ROOT = Path(
    f"/content/ebd_{EVENT_NAME}_local"
)

ZIP_PATH = (
    LOCAL_ROOT /
    "downloads" /
    "HURRICANE-DORIAN.zip"
)

EXTRACT_DIR = (
    LOCAL_ROOT /
    "extracted"
)

EXTRACT_MARKER = (
    EXTRACT_DIR /
    ".extraction_completed"
)

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 2. EXTRACT ZIP
# ------------------------------------------------------------

def extract_event_zip(
    zip_path,
    extract_dir,
):
    zip_path = Path(zip_path)
    extract_dir = Path(extract_dir)

    if not zip_path.is_file():
        raise FileNotFoundError(
            f"ZIP file not found:\n{zip_path}"
        )

    if not zipfile.is_zipfile(zip_path):
        raise zipfile.BadZipFile(
            f"Invalid ZIP file:\n{zip_path}"
        )

    existing_png_files = list(
        extract_dir.rglob("*.png")
    )

    # اگر قبلاً کامل استخراج شده، استخراج تکرار نمی‌شود
    if (
        EXTRACT_MARKER.exists()
        and len(existing_png_files) > 0
    ):
        print("Extraction marker found.")
        print("Existing PNG files:", len(existing_png_files))
        print("Skipping repeated extraction.")

        return extract_dir

    print("=" * 72)
    print("EXTRACTING HURRICANE-DORIAN")
    print("=" * 72)
    print("ZIP        :", zip_path)
    print("Destination:", extract_dir)

    with zipfile.ZipFile(zip_path, "r") as archive:
        members = archive.infolist()

        for member in tqdm(
            members,
            desc="Extracting Dorian ZIP",
            unit="file",
        ):
            archive.extract(
                member,
                path=extract_dir,
            )

    EXTRACT_MARKER.write_text(
        "HURRICANE-DORIAN extraction completed.",
        encoding="utf-8",
    )

    print("\nExtraction completed successfully.")
    print("Destination:", extract_dir)

    return extract_dir


# ------------------------------------------------------------
# 3. CREATE FILE INDEX
# ------------------------------------------------------------

def create_event_file_indexes(
    extract_dir,
):
    extract_dir = Path(extract_dir)

    all_png_files = sorted(
        extract_dir.rglob("*.png")
    )

    image_files = [
        path
        for path in all_png_files
        if path.parent.name.lower() == "images"
    ]

    mask_files = [
        path
        for path in all_png_files
        if path.parent.name.lower() == "masks"
    ]

    pre_images = {
        path.name.replace(
            "_pre_disaster.png",
            "",
        ): path
        for path in image_files
        if path.name.endswith(
            "_pre_disaster.png"
        )
    }

    post_images = {
        path.name.replace(
            "_post_disaster.png",
            "",
        ): path
        for path in image_files
        if path.name.endswith(
            "_post_disaster.png"
        )
    }

    pre_masks = {
        path.name.replace(
            "_pre_disaster.png",
            "",
        ): path
        for path in mask_files
        if path.name.endswith(
            "_pre_disaster.png"
        )
    }

    post_masks = {
        path.name.replace(
            "_post_disaster.png",
            "",
        ): path
        for path in mask_files
        if path.name.endswith(
            "_post_disaster.png"
        )
    }

    indexes = {
        "pre_image": pre_images,
        "post_image": post_images,
        "pre_mask": pre_masks,
        "post_mask": post_masks,
    }

    print("\n" + "=" * 72)
    print("EXTRACTED FILE INVENTORY")
    print("=" * 72)
    print("Total PNG files:", len(all_png_files))
    print("Pre images     :", len(pre_images))
    print("Post images    :", len(post_images))
    print("Pre masks      :", len(pre_masks))
    print("Post masks     :", len(post_masks))
    print("=" * 72)

    return indexes


# ------------------------------------------------------------
# 4. VERIFY COMPLETE PAIRS
# ------------------------------------------------------------

def verify_complete_event_pairs(
    indexes,
):
    all_scene_ids = set()

    for file_index in indexes.values():
        all_scene_ids.update(
            file_index.keys()
        )

    complete_pairs = []
    incomplete_pairs = []

    for scene_id in sorted(all_scene_ids):
        missing_parts = [
            component
            for component, file_index in indexes.items()
            if scene_id not in file_index
        ]

        if len(missing_parts) == 0:
            complete_pairs.append({
                "scene_id":
                    scene_id,

                "pre_image":
                    indexes["pre_image"][scene_id],

                "post_image":
                    indexes["post_image"][scene_id],

                "pre_mask":
                    indexes["pre_mask"][scene_id],

                "post_mask":
                    indexes["post_mask"][scene_id],
            })

        else:
            incomplete_pairs.append({
                "scene_id": scene_id,
                "missing": missing_parts,
            })

    print("\n" + "=" * 72)
    print("PRE/POST PAIRING CHECK")
    print("=" * 72)
    print("Unique scene IDs :", len(all_scene_ids))
    print("Complete pairs   :", len(complete_pairs))
    print("Incomplete pairs :", len(incomplete_pairs))

    if incomplete_pairs:
        print("\nFirst incomplete pairs:")

        for item in incomplete_pairs[:10]:
            print(
                item["scene_id"],
                "missing:",
                item["missing"],
            )

    if complete_pairs:
        example = complete_pairs[0]

        print("\nExample complete pair:")
        print("Scene ID  :", example["scene_id"])
        print("Pre image :", example["pre_image"])
        print("Post image:", example["post_image"])
        print("Pre mask  :", example["pre_mask"])
        print("Post mask :", example["post_mask"])

    print("=" * 72)

    if len(complete_pairs) == 0:
        raise RuntimeError(
            "No complete pre/post pair was found."
        )

    if len(incomplete_pairs) > 0:
        raise RuntimeError(
            "Incomplete pre/post pairs were detected."
        )

    return complete_pairs


# ------------------------------------------------------------
# 5. VERIFY EVENT IDENTIFIERS
# ------------------------------------------------------------

def verify_event_names(
    complete_pairs,
    expected_event,
):
    invalid_scene_ids = [
        pair["scene_id"]
        for pair in complete_pairs
        if not pair["scene_id"].upper().startswith(
            expected_event.upper()
        )
    ]

    print("\nEvent-name validation:")
    print("Expected event    :", expected_event)
    print("Invalid scene IDs :", len(invalid_scene_ids))

    if invalid_scene_ids:
        print("Examples:", invalid_scene_ids[:10])

        raise RuntimeError(
            "Some extracted scenes do not belong to "
            f"{expected_event}."
        )

    print("✅ All scene IDs belong to the requested event.")


# ------------------------------------------------------------
# 6. RUN STEP 2
# ------------------------------------------------------------

extracted_root = extract_event_zip(
    zip_path=ZIP_PATH,
    extract_dir=EXTRACT_DIR,
)

event_indexes = create_event_file_indexes(
    extract_dir=extracted_root,
)

complete_pairs = verify_complete_event_pairs(
    indexes=event_indexes,
)

verify_event_names(
    complete_pairs=complete_pairs,
    expected_event=EVENT_NAME,
)

print("\n" + "=" * 72)
print("STEP 2 COMPLETED SUCCESSFULLY")
print("=" * 72)
print("Extracted root :", extracted_root)
print("Complete pairs:", len(complete_pairs))
print("=" * 72)

EXTRACTING HURRICANE-DORIAN
ZIP        : /content/ebd_HURRICANE-DORIAN_local/downloads/HURRICANE-DORIAN.zip
Destination: /content/ebd_HURRICANE-DORIAN_local/extracted


Extracting Dorian ZIP:   0%|          | 0/2163 [00:00<?, ?file/s]


Extraction completed successfully.
Destination: /content/ebd_HURRICANE-DORIAN_local/extracted

EXTRACTED FILE INVENTORY
Total PNG files: 2160
Pre images     : 540
Post images    : 540
Pre masks      : 540
Post masks     : 540

PRE/POST PAIRING CHECK
Unique scene IDs : 540
Complete pairs   : 540
Incomplete pairs : 0

Example complete pair:
Scene ID  : HURRICANE-DORIAN_000011
Pre image : /content/ebd_HURRICANE-DORIAN_local/extracted/HURRICANE-DORIAN/images/HURRICANE-DORIAN_000011_pre_disaster.png
Post image: /content/ebd_HURRICANE-DORIAN_local/extracted/HURRICANE-DORIAN/images/HURRICANE-DORIAN_000011_post_disaster.png
Pre mask  : /content/ebd_HURRICANE-DORIAN_local/extracted/HURRICANE-DORIAN/masks/HURRICANE-DORIAN_000011_pre_disaster.png
Post mask : /content/ebd_HURRICANE-DORIAN_local/extracted/HURRICANE-DORIAN/masks/HURRICANE-DORIAN_000011_post_disaster.png

Event-name validation:
Expected event    : HURRICANE-DORIAN
Invalid scene IDs : 0
✅ All scene IDs belong to the requested event.


In [35]:
# ============================================================
# STEP 3 — CREATE MATCHED PRE/POST 512-VALID TEST PATCHES
# ============================================================

import os
import glob
import shutil
import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter, defaultdict


# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

EVENT_NAME = "HURRICANE-DORIAN"

LOCAL_ROOT = Path(
    f"/content/ebd_{EVENT_NAME}_local"
)

EXTRACT_DIR = (
    LOCAL_ROOT /
    "extracted"
)

PAIRED_TEST_ROOT = (
    LOCAL_ROOT /
    f"{EVENT_NAME}_prepost_npz_512valid_as_256"
)

MANIFEST_PATH = (
    PAIRED_TEST_ROOT /
    "test_manifest.csv"
)

PATCH_SIZE = 256
BLOCK_SIZE = 512

NUM_CLASSES = 4
REQUIRED_DAMAGE_CLASSES = {1, 2, 3}

# در اولین اجرا False بماند.
# فقط اگر قصد بازسازی کامل پوشه را داری True کن.
RESET_OUTPUT = False


# ------------------------------------------------------------
# 2. SAFE OUTPUT PREPARATION
# ------------------------------------------------------------

if RESET_OUTPUT and PAIRED_TEST_ROOT.is_dir():
    print("Removing previous paired test directory:")
    print(PAIRED_TEST_ROOT)

    shutil.rmtree(PAIRED_TEST_ROOT)

PAIRED_TEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

existing_npz_files = sorted(
    PAIRED_TEST_ROOT.glob("*.npz")
)

if existing_npz_files and not RESET_OUTPUT:
    raise RuntimeError(
        f"{len(existing_npz_files)} existing NPZ files were found in:\n"
        f"{PAIRED_TEST_ROOT}\n\n"
        "To prevent mixing old and new test data, either use the "
        "existing completed dataset or set RESET_OUTPUT=True."
    )


# ------------------------------------------------------------
# 3. DAMAGE-LABEL CONVERSION
# ------------------------------------------------------------

def convert_damage_to_3class_plus_background(
    original_damage,
):
    """
    Original EBD
    ------------
    0 = background
    1 = no damage
    2 = minor damage
    3 = major damage
    4 = destroyed

    STRATA classes
    --------------
    0 = background
    1 = intact = no damage + minor
    2 = damage = major damage
    3 = destroyed
    """
    converted = np.zeros_like(
        original_damage,
        dtype=np.uint8,
    )

    converted[
        (original_damage == 1)
        | (original_damage == 2)
    ] = 1

    converted[
        original_damage == 3
    ] = 2

    converted[
        original_damage == 4
    ] = 3

    return converted


# ------------------------------------------------------------
# 4. VALID 512-BLOCK CONDITION
# ------------------------------------------------------------

def block_has_required_classes(
    damage_block,
    building_block,
    required_classes,
):
    valid_building = building_block > 0

    if not np.any(valid_building):
        return False

    classes_inside_buildings = set(
        np.unique(
            damage_block[valid_building]
        ).astype(int).tolist()
    )

    return required_classes.issubset(
        classes_inside_buildings
    )


# ------------------------------------------------------------
# 5. READ AND CHECK A COMPLETE SCENE
# ------------------------------------------------------------

def read_complete_scene(pair):
    pre_image = cv2.imread(
        str(pair["pre_image"]),
        cv2.IMREAD_COLOR,
    )

    post_image = cv2.imread(
        str(pair["post_image"]),
        cv2.IMREAD_COLOR,
    )

    pre_mask = cv2.imread(
        str(pair["pre_mask"]),
        cv2.IMREAD_GRAYSCALE,
    )

    post_mask = cv2.imread(
        str(pair["post_mask"]),
        cv2.IMREAD_GRAYSCALE,
    )

    arrays = {
        "pre_image": pre_image,
        "post_image": post_image,
        "pre_mask": pre_mask,
        "post_mask": post_mask,
    }

    missing = [
        name
        for name, array in arrays.items()
        if array is None
    ]

    if missing:
        raise RuntimeError(
            f"Could not read {missing} for scene "
            f"{pair['scene_id']}"
        )

    spatial_shapes = {
        pre_image.shape[:2],
        post_image.shape[:2],
        pre_mask.shape,
        post_mask.shape,
    }

    if len(spatial_shapes) != 1:
        raise RuntimeError(
            f"Spatial-shape mismatch for scene "
            f"{pair['scene_id']}:\n"
            f"pre_image={pre_image.shape}, "
            f"post_image={post_image.shape}, "
            f"pre_mask={pre_mask.shape}, "
            f"post_mask={post_mask.shape}"
        )

    return (
        pre_image,
        post_image,
        pre_mask,
        post_mask,
    )


# ------------------------------------------------------------
# 6. CREATE PAIRED TEST PATCHES
# ------------------------------------------------------------

def create_paired_512valid_test_patches(
    complete_pairs,
):
    if not complete_pairs:
        raise RuntimeError(
            "complete_pairs is empty. Run Step 2 first."
        )

    saved_blocks = 0
    saved_patches = 0

    skipped_missing_file = 0
    skipped_bad_shape = 0
    skipped_incomplete_512 = 0
    skipped_no_building = 0
    skipped_missing_classes = 0
    skipped_black_pre = 0
    skipped_black_post = 0

    original_damage_values = Counter()
    converted_damage_values = Counter()

    manifest_rows = []

    for pair in tqdm(
        complete_pairs,
        desc="Preparing paired Dorian test data",
        unit="scene",
    ):
        scene_id = pair["scene_id"]

        try:
            (
                pre_image,
                post_image,
                pre_mask,
                post_mask,
            ) = read_complete_scene(pair)

        except FileNotFoundError:
            skipped_missing_file += 1
            continue

        except RuntimeError as error:
            print("\nWARNING:", error)
            skipped_bad_shape += 1
            continue

        building_mask = (
            pre_mask > 0
        ).astype(np.uint8)

        damage_mask = (
            convert_damage_to_3class_plus_background(
                post_mask
            )
        )

        for value, count in zip(
            *np.unique(
                post_mask,
                return_counts=True,
            )
        ):
            original_damage_values[int(value)] += int(count)

        for value, count in zip(
            *np.unique(
                damage_mask,
                return_counts=True,
            )
        ):
            converted_damage_values[int(value)] += int(count)

        height, width = damage_mask.shape

        for block_y in range(
            0,
            height,
            BLOCK_SIZE,
        ):
            for block_x in range(
                0,
                width,
                BLOCK_SIZE,
            ):
                pre_block = pre_image[
                    block_y:block_y + BLOCK_SIZE,
                    block_x:block_x + BLOCK_SIZE,
                ]

                post_block = post_image[
                    block_y:block_y + BLOCK_SIZE,
                    block_x:block_x + BLOCK_SIZE,
                ]

                building_block = building_mask[
                    block_y:block_y + BLOCK_SIZE,
                    block_x:block_x + BLOCK_SIZE,
                ]

                damage_block = damage_mask[
                    block_y:block_y + BLOCK_SIZE,
                    block_x:block_x + BLOCK_SIZE,
                ]

                expected_image_shape = (
                    BLOCK_SIZE,
                    BLOCK_SIZE,
                    3,
                )

                expected_mask_shape = (
                    BLOCK_SIZE,
                    BLOCK_SIZE,
                )

                if (
                    pre_block.shape != expected_image_shape
                    or post_block.shape != expected_image_shape
                    or building_block.shape != expected_mask_shape
                    or damage_block.shape != expected_mask_shape
                ):
                    skipped_incomplete_512 += 1
                    continue

                if not np.any(building_block > 0):
                    skipped_no_building += 1
                    continue

                if not block_has_required_classes(
                    damage_block=damage_block,
                    building_block=building_block,
                    required_classes=REQUIRED_DAMAGE_CLASSES,
                ):
                    skipped_missing_classes += 1
                    continue

                # برای گزارش فقط؛ مطابق STRATA این بلوک‌ها حذف نمی‌شوند.
                if np.all(pre_block == 0):
                    skipped_black_pre += 1

                if np.all(post_block == 0):
                    skipped_black_post += 1

                # همان چهار موقعیت کد STRATA
                patch_positions = [
                    (0, 0),
                    (0, PATCH_SIZE),
                    (PATCH_SIZE, 0),
                    (PATCH_SIZE, PATCH_SIZE),
                ]

                block_patch_count = 0

                for local_y, local_x in patch_positions:
                    y = block_y + local_y
                    x = block_x + local_x

                    pre_patch = pre_image[
                        y:y + PATCH_SIZE,
                        x:x + PATCH_SIZE,
                    ]

                    post_patch = post_image[
                        y:y + PATCH_SIZE,
                        x:x + PATCH_SIZE,
                    ]

                    building_patch = building_mask[
                        y:y + PATCH_SIZE,
                        x:x + PATCH_SIZE,
                    ]

                    damage_patch = damage_mask[
                        y:y + PATCH_SIZE,
                        x:x + PATCH_SIZE,
                    ]

                    if (
                        pre_patch.shape
                        != (PATCH_SIZE, PATCH_SIZE, 3)
                        or post_patch.shape
                        != (PATCH_SIZE, PATCH_SIZE, 3)
                        or building_patch.shape
                        != (PATCH_SIZE, PATCH_SIZE)
                        or damage_patch.shape
                        != (PATCH_SIZE, PATCH_SIZE)
                    ):
                        raise RuntimeError(
                            f"Unexpected 256-patch shape: "
                            f"{scene_id}, y={y}, x={x}"
                        )

                    save_name = (
                        f"{scene_id}_{y}_{x}.npz"
                    )

                    save_path = (
                        PAIRED_TEST_ROOT /
                        save_name
                    )

                    np.savez_compressed(
                        save_path,

                        # Siamese U-Net inputs
                        pre_image=pre_patch.astype(
                            np.uint8
                        ),

                        post_image=post_patch.astype(
                            np.uint8
                        ),

                        # Same STRATA quantitative evaluation targets
                        building_mask=building_patch.astype(
                            np.uint8
                        ),

                        damage_mask=damage_patch.astype(
                            np.uint8
                        ),
                    )

                    classes_in_patch = sorted(
                        np.unique(
                            damage_patch[
                                building_patch > 0
                            ]
                        ).astype(int).tolist()
                    ) if np.any(building_patch > 0) else []

                    manifest_rows.append({
                        "event":
                            EVENT_NAME,

                        "scene_id":
                            scene_id,

                        "block_y":
                            block_y,

                        "block_x":
                            block_x,

                        "patch_y":
                            y,

                        "patch_x":
                            x,

                        "filename":
                            save_name,

                        "valid_building_pixels":
                            int(
                                np.sum(
                                    building_patch > 0
                                )
                            ),

                        "classes_inside_building":
                            ",".join(
                                map(str, classes_in_patch)
                            ),

                        "block_contains_all_classes":
                            True,
                    })

                    saved_patches += 1
                    block_patch_count += 1

                if block_patch_count != 4:
                    raise RuntimeError(
                        f"Block {scene_id}, "
                        f"y={block_y}, x={block_x} "
                        f"did not produce exactly four patches."
                    )

                saved_blocks += 1

    manifest_df = pd.DataFrame(
        manifest_rows
    )

    manifest_df.to_csv(
        MANIFEST_PATH,
        index=False,
    )

    print("\n" + "=" * 72)
    print("PAIRED DORIAN TEST PREPARATION SUMMARY")
    print("=" * 72)
    print("Complete scenes examined :", len(complete_pairs))
    print("Valid 512 blocks         :", saved_blocks)
    print("Saved 256 patches        :", saved_patches)
    print("Expected patches         :", saved_blocks * 4)
    print("Skipped missing file     :", skipped_missing_file)
    print("Skipped bad shape        :", skipped_bad_shape)
    print("Skipped incomplete 512   :", skipped_incomplete_512)
    print("Skipped no building      :", skipped_no_building)
    print("Skipped missing classes  :", skipped_missing_classes)
    print("All-black pre blocks     :", skipped_black_pre)
    print("All-black post blocks    :", skipped_black_post)
    print("Output directory         :", PAIRED_TEST_ROOT)
    print("Manifest                 :", MANIFEST_PATH)

    print("\nOriginal post-mask pixel counts:")
    print(dict(sorted(original_damage_values.items())))

    print("\nConverted damage-mask pixel counts:")
    print(dict(sorted(converted_damage_values.items())))

    if saved_blocks == 0:
        raise RuntimeError(
            "No 512 block containing all classes "
            "{1, 2, 3} was found."
        )

    if saved_patches != saved_blocks * 4:
        raise RuntimeError(
            "Saved patch count is inconsistent with "
            "the STRATA 512-valid protocol."
        )

    print("\n✅ Matched paired test data created successfully.")
    print("=" * 72)

    return {
        "root": str(PAIRED_TEST_ROOT),
        "manifest": str(MANIFEST_PATH),
        "valid_512_blocks": saved_blocks,
        "saved_256_patches": saved_patches,
    }


# ------------------------------------------------------------
# 7. RUN STEP 3
# ------------------------------------------------------------

paired_test_report = (
    create_paired_512valid_test_patches(
        complete_pairs=complete_pairs,
    )
)

paired_test_report

Preparing paired Dorian test data:   0%|          | 0/540 [00:00<?, ?scene/s]


PAIRED DORIAN TEST PREPARATION SUMMARY
Complete scenes examined : 540
Valid 512 blocks         : 181
Saved 256 patches        : 724
Expected patches         : 724
Skipped missing file     : 0
Skipped bad shape        : 0
Skipped incomplete 512   : 0
Skipped no building      : 170
Skipped missing classes  : 189
All-black pre blocks     : 0
All-black post blocks    : 0
Output directory         : /content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256
Manifest                 : /content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256/test_manifest.csv

Original post-mask pixel counts:
{0: 135671818, 1: 564023, 2: 1692972, 3: 2363588, 4: 1265359}

Converted damage-mask pixel counts:
{0: 135671818, 1: 2256995, 2: 2363588, 3: 1265359}

✅ Matched paired test data created successfully.


{'root': '/content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256',
 'manifest': '/content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256/test_manifest.csv',
 'valid_512_blocks': 181,
 'saved_256_patches': 724}

In [36]:
# ============================================================
# STEP 3.1 — VALIDATE GENERATED PAIRED NPZ FILES
# ============================================================

import random
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm


def validate_paired_test_npz(
    root_dir,
    full_validation=True,
    random_display_count=5,
    seed=42,
):
    root_dir = Path(root_dir)

    files = sorted(
        root_dir.glob("*.npz")
    )

    if len(files) == 0:
        raise FileNotFoundError(
            f"No paired test NPZ files found in:\n{root_dir}"
        )

    required_keys = {
        "pre_image",
        "post_image",
        "building_mask",
        "damage_mask",
    }

    validation_files = (
        files
        if full_validation
        else files[:min(20, len(files))]
    )

    errors = []

    for path in tqdm(
        validation_files,
        desc="Validating paired test NPZ",
        unit="file",
    ):
        try:
            with np.load(path) as data:
                keys = set(data.keys())

                if keys != required_keys:
                    errors.append(
                        f"{path.name}: keys={sorted(keys)}"
                    )
                    continue

                pre_image = data["pre_image"]
                post_image = data["post_image"]
                building = data["building_mask"]
                damage = data["damage_mask"]

                if pre_image.shape != (256, 256, 3):
                    errors.append(
                        f"{path.name}: invalid pre shape "
                        f"{pre_image.shape}"
                    )

                if post_image.shape != (256, 256, 3):
                    errors.append(
                        f"{path.name}: invalid post shape "
                        f"{post_image.shape}"
                    )

                if building.shape != (256, 256):
                    errors.append(
                        f"{path.name}: invalid building shape "
                        f"{building.shape}"
                    )

                if damage.shape != (256, 256):
                    errors.append(
                        f"{path.name}: invalid damage shape "
                        f"{damage.shape}"
                    )

                if pre_image.dtype != np.uint8:
                    errors.append(
                        f"{path.name}: pre dtype={pre_image.dtype}"
                    )

                if post_image.dtype != np.uint8:
                    errors.append(
                        f"{path.name}: post dtype={post_image.dtype}"
                    )

                if not set(
                    np.unique(building).tolist()
                ).issubset({0, 1}):
                    errors.append(
                        f"{path.name}: invalid building values"
                    )

                if not set(
                    np.unique(damage).tolist()
                ).issubset({0, 1, 2, 3}):
                    errors.append(
                        f"{path.name}: invalid damage values"
                    )

        except Exception as error:
            errors.append(
                f"{path.name}: {repr(error)}"
            )

    print("\n" + "=" * 72)
    print("PAIRED NPZ VALIDATION")
    print("=" * 72)
    print("Files found     :", len(files))
    print("Files validated :", len(validation_files))
    print("Errors          :", len(errors))

    if errors:
        print("\nFirst errors:")

        for error in errors[:20]:
            print(" ", error)

        raise RuntimeError(
            "Paired test NPZ validation failed."
        )

    if len(files) % 4 != 0:
        raise RuntimeError(
            "NPZ count is not divisible by four."
        )

    random_generator = random.Random(seed)

    selected_files = random_generator.sample(
        files,
        k=min(random_display_count, len(files)),
    )

    print("\nRandom valid samples:")

    for path in selected_files:
        with np.load(path) as data:
            print(
                f"  {path.name} | "
                f"damage={np.unique(data['damage_mask']).tolist()} | "
                f"building_pixels="
                f"{int(np.sum(data['building_mask'] > 0))}"
            )

    print("\nComplete 512 groups:", len(files) // 4)
    print("✅ All paired test NPZ files are valid.")
    print("=" * 72)

    return files


paired_test_files = validate_paired_test_npz(
    root_dir=PAIRED_TEST_ROOT,
    full_validation=True,
    random_display_count=5,
    seed=42,
)

Validating paired test NPZ:   0%|          | 0/724 [00:00<?, ?file/s]


PAIRED NPZ VALIDATION
Files found     : 724
Files validated : 724
Errors          : 0

Random valid samples:
  HURRICANE-DORIAN_016657_256_0.npz | damage=[0, 1, 2, 3] | building_pixels=11387
  HURRICANE-DORIAN_003221_256_0.npz | damage=[0, 1, 2, 3] | building_pixels=8442
  HURRICANE-DORIAN_000151_0_256.npz | damage=[0, 1, 2, 3] | building_pixels=2752
  HURRICANE-DORIAN_007846_0_256.npz | damage=[0, 1, 2] | building_pixels=4421
  HURRICANE-DORIAN_006688_256_0.npz | damage=[0, 1, 2, 3] | building_pixels=2606

Complete 512 groups: 181
✅ All paired test NPZ files are valid.


In [38]:
# ============================================================
# STEP 4.1 — TEST SETTINGS AND CHECKPOINT DISCOVERY
# ============================================================

import os
import re
import glob
import torch
import numpy as np

from pathlib import Path


# ------------------------------------------------------------
# 1. TEST SETTINGS
# ------------------------------------------------------------

EVENT_NAME = "HURRICANE-DORIAN"
EVENT_TAG = re.sub(
    r"[^A-Za-z0-9]+",
    "_",
    EVENT_NAME,
).strip("_")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

NUM_CLASSES = 4
CLASS_NAMES = [
    "background",
    "intact",
    "damage",
    "destroyed",
]

FG_CLASS_NAMES = [
    "intact",
    "damage",
    "destroyed",
]

TEST_BATCH_SIZE = 16

# به علت خطای قبلی DataLoader
TEST_NUM_WORKERS = 0

PIN_MEMORY = torch.cuda.is_available()


# ------------------------------------------------------------
# 2. INPUT DATA
# ------------------------------------------------------------

PAIRED_TEST_ROOT = Path(
    f"/content/ebd_{EVENT_NAME}_local/"
    f"{EVENT_NAME}_prepost_npz_512valid_as_256"
)


# ------------------------------------------------------------
# 3. MODEL RUN WORKSPACE
# ------------------------------------------------------------

SIAMESE_RUN_ROOT = Path(
    "/content/drive/MyDrive/STRATA_runs/"
    "PrePost_Baselines/"
    "ChangeOS_R18/"
    "budget500_seed42"
)

MODEL_ROOT = (
    SIAMESE_RUN_ROOT /
    "models"
)

TEST_OUTPUT_DIR = (
    SIAMESE_RUN_ROOT /
    f"{EVENT_TAG}_test_metrics"
)

TEST_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 4. CHECKPOINT DISCOVERY
# ------------------------------------------------------------

def discover_damage_checkpoints(
    model_root,
):
    model_root = Path(model_root)

    if not model_root.is_dir():
        raise FileNotFoundError(
            f"Model directory not found:\n{model_root}"
        )

    checkpoint_extensions = {
        ".pth",
        ".pt",
        ".ckpt",
    }

    all_checkpoints = sorted(
        path
        for path in model_root.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() in checkpoint_extensions
        )
    )

    if len(all_checkpoints) == 0:
        raise FileNotFoundError(
            f"No checkpoint was found under:\n{model_root}"
        )

    print("=" * 72)
    print("AVAILABLE SIAMESE U-NET CHECKPOINTS")
    print("=" * 72)

    for index, path in enumerate(
        all_checkpoints,
        start=1,
    ):
        size_mb = path.stat().st_size / 1024**2

        print(
            f"{index:02d}. {path} "
            f"({size_mb:.2f} MB)"
        )

    # اولویت با best checkpoint مربوط به damage
    damage_candidates = [
        path
        for path in all_checkpoints
        if (
            "damage" in str(path).lower()
            and "best" in path.name.lower()
        )
    ]

    if len(damage_candidates) == 0:
        damage_candidates = [
            path
            for path in all_checkpoints
            if "damage" in str(path).lower()
        ]

    if len(damage_candidates) == 0:
        raise FileNotFoundError(
            "No damage-stage checkpoint was found. "
            "Do not use the localization checkpoint for testing."
        )

    # اگر چند فایل بود، جدیدترین best damage انتخاب می‌شود
    damage_candidates = sorted(
        damage_candidates,
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    selected_checkpoint = damage_candidates[0]

    print("\nSelected damage checkpoint:")
    print(selected_checkpoint)
    print("=" * 72)

    return selected_checkpoint


BEST_DAMAGE_CHECKPOINT = discover_damage_checkpoints(
    MODEL_ROOT
)

print("\nTest configuration:")
print("Device          :", DEVICE)
print("Test data       :", PAIRED_TEST_ROOT)
print("Checkpoint      :", BEST_DAMAGE_CHECKPOINT)
print("Test output     :", TEST_OUTPUT_DIR)
print("Test batch size :", TEST_BATCH_SIZE)
print("Test workers    :", TEST_NUM_WORKERS)

AVAILABLE SIAMESE U-NET CHECKPOINTS
01. /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models/damage/best_changeos_r18.pth (289.33 MB)
02. /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models/damage/latest_changeos_r18.pth (289.34 MB)

Selected damage checkpoint:
/content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models/damage/best_changeos_r18.pth

Test configuration:
Device          : cuda
Test data       : /content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256
Checkpoint      : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models/damage/best_changeos_r18.pth
Test output     : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/HURRICANE_DORIAN_test_metrics
Test batch size : 16
Test workers    : 0


In [39]:
# ============================================================
# STEP 4.2 — INDEPENDENT PAIRED TEST DATASET
# ============================================================

import os
import glob
import numpy as np
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader


# ------------------------------------------------------------
# 1. BASIC UTILITIES
# ------------------------------------------------------------

def normalize_test_image(image):
    """
    Same normalization used during Siamese U-Net training.
    BGR channel order is preserved.
    """
    return image.astype(
        np.float32
    ) / 255.0


def safe_contiguous(array):
    return np.ascontiguousarray(
        array
    )


def get_event_from_filename(path):
    return Path(path).name.split("_")[0]


# ------------------------------------------------------------
# 2. DATASET
# ------------------------------------------------------------

class SiameseIndependentTestDataset(Dataset):

    def __init__(
        self,
        root_dir,
    ):
        self.root_dir = Path(root_dir)

        self.files = sorted(
            self.root_dir.glob("*.npz")
        )

        if len(self.files) == 0:
            raise FileNotFoundError(
                f"No test NPZ files found in:\n"
                f"{self.root_dir}"
            )

    def __len__(self):
        return len(self.files)

    def __getitem__(
        self,
        index,
    ):
        file_path = self.files[index]

        with np.load(file_path) as data:
            pre_image = np.array(
                data["pre_image"],
                dtype=np.uint8,
                copy=True,
            )

            post_image = np.array(
                data["post_image"],
                dtype=np.uint8,
                copy=True,
            )

            damage_mask = np.array(
                data["damage_mask"],
                dtype=np.uint8,
                copy=True,
            )

            building_mask = np.array(
                data["building_mask"],
                dtype=np.uint8,
                copy=True,
            )

        pre_image = normalize_test_image(
            pre_image
        )

        post_image = normalize_test_image(
            post_image
        )

        # HWC -> CHW
        pre_image = np.transpose(
            pre_image,
            (2, 0, 1),
        )

        post_image = np.transpose(
            post_image,
            (2, 0, 1),
        )

        pre_image = safe_contiguous(
            pre_image
        )

        post_image = safe_contiguous(
            post_image
        )

        damage_mask = safe_contiguous(
            damage_mask
        )

        building_mask = safe_contiguous(
            building_mask
        )

        pre_tensor = torch.from_numpy(
            pre_image
        ).float()

        post_tensor = torch.from_numpy(
            post_image
        ).float()

        damage_tensor = torch.from_numpy(
            damage_mask
        ).long()

        building_tensor = torch.from_numpy(
            building_mask
        ).float()

        event_name = get_event_from_filename(
            file_path
        )

        return (
            pre_tensor,
            post_tensor,
            damage_tensor,
            building_tensor,
            event_name,
            str(file_path),
        )


# ------------------------------------------------------------
# 3. DATA LOADER
# ------------------------------------------------------------

test_dataset = SiameseIndependentTestDataset(
    root_dir=PAIRED_TEST_ROOT
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=TEST_NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

print("=" * 72)
print("SIAMESE INDEPENDENT TEST DATA")
print("=" * 72)
print("Event       :", EVENT_NAME)
print("Test patches:", len(test_dataset))
print("Test batches:", len(test_loader))
print("First file  :", test_dataset.files[0])
print("Last file   :", test_dataset.files[-1])
print("=" * 72)

assert len(test_dataset) == 724, (
    f"Expected 724 test patches, "
    f"but found {len(test_dataset)}."
)

SIAMESE INDEPENDENT TEST DATA
Event       : HURRICANE-DORIAN
Test patches: 724
Test batches: 46
First file  : /content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256/HURRICANE-DORIAN_000011_0_0.npz
Last file   : /content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256/HURRICANE-DORIAN_017989_256_256.npz


In [43]:
# ============================================================
# STEP 4.3 — LOAD BEST CHANGEOS-R18 CHECKPOINT
# ============================================================

import torch

from pathlib import Path
from collections import OrderedDict


# ============================================================
# 1. EXTRACT MODEL STATE DICTIONARY
# ============================================================

def extract_model_state_dict(
    checkpoint,
):
    """
    Supported structures
    --------------------
    checkpoint["model_state_dict"]
    checkpoint["model"]
    checkpoint["state_dict"]
    checkpoint["net"]
    checkpoint["network"]
    direct state_dict
    """
    if not isinstance(
        checkpoint,
        dict,
    ):
        raise TypeError(
            "Checkpoint must be a dictionary."
        )

    candidate_keys = [
        "model_state_dict",
        "model",
        "state_dict",
        "net",
        "network",
    ]

    for key in candidate_keys:
        if (
            key in checkpoint
            and isinstance(
                checkpoint[key],
                dict,
            )
        ):
            print(
                "Model state dictionary found in "
                f"checkpoint['{key}']."
            )

            return checkpoint[key]

    is_direct_state_dictionary = (
        len(checkpoint) > 0
        and all(
            isinstance(key, str)
            for key in checkpoint.keys()
        )
        and any(
            torch.is_tensor(value)
            for value in checkpoint.values()
        )
    )

    if is_direct_state_dictionary:
        print(
            "Checkpoint itself is a direct state dictionary."
        )

        return checkpoint

    raise KeyError(
        "Could not find a model state dictionary "
        "inside the checkpoint."
    )


# ============================================================
# 2. REMOVE OPTIONAL DATAPARALLEL PREFIX
# ============================================================

def remove_module_prefix(
    state_dict,
):
    cleaned_state_dict = OrderedDict()

    for key, value in state_dict.items():
        cleaned_key = (
            key[7:]
            if key.startswith("module.")
            else key
        )

        cleaned_state_dict[
            cleaned_key
        ] = value

    return cleaned_state_dict


# ============================================================
# 3. PRINT CHECKPOINT METADATA
# ============================================================

def print_checkpoint_metadata(
    checkpoint,
):
    if not isinstance(
        checkpoint,
        dict,
    ):
        return

    metadata_keys = [
        "model_name",
        "run_name",
        "epoch",
        "best_epoch",
        "best_score",
        "selection_metric",
        "num_damage_classes",
        "decoder_channels",
        "imagenet_pretrained",
        "input_is_bgr",
        "learning_rate",
        "loss",
    ]

    print("\nCheckpoint metadata:")

    metadata_found = False

    for key in metadata_keys:
        if key in checkpoint:
            value = checkpoint[key]

            if (
                isinstance(
                    value,
                    (
                        str,
                        int,
                        float,
                        bool,
                    ),
                )
                or value is None
            ):
                print(
                    f"  {key}: {value}"
                )

                metadata_found = True

    if not metadata_found:
        print(
            "  No simple metadata fields found."
        )


# ============================================================
# 4. VERIFY CHECKPOINT IDENTITY
# ============================================================

def verify_changeos_checkpoint(
    checkpoint,
):
    if not isinstance(
        checkpoint,
        dict,
    ):
        return

    checkpoint_model_name = (
        checkpoint.get(
            "model_name",
            None,
        )
    )

    if (
        checkpoint_model_name is not None
        and checkpoint_model_name
        != MODEL_NAME
    ):
        raise RuntimeError(
            "Wrong model checkpoint detected.\n"
            f"Checkpoint model: {checkpoint_model_name}\n"
            f"Expected model:   {MODEL_NAME}"
        )

    checkpoint_classes = (
        checkpoint.get(
            "num_damage_classes",
            None,
        )
    )

    if (
        checkpoint_classes is not None
        and int(checkpoint_classes)
        != NUM_DAMAGE_CLASSES
    ):
        raise RuntimeError(
            "Damage-class mismatch.\n"
            f"Checkpoint classes: {checkpoint_classes}\n"
            f"Expected classes:   {NUM_DAMAGE_CLASSES}"
        )

    checkpoint_decoder_channels = (
        checkpoint.get(
            "decoder_channels",
            None,
        )
    )

    if (
        checkpoint_decoder_channels
        is not None
        and int(checkpoint_decoder_channels)
        != CHANGEOS_DECODER_CHANNELS
    ):
        raise RuntimeError(
            "Decoder-channel mismatch.\n"
            f"Checkpoint: {checkpoint_decoder_channels}\n"
            f"Expected:   {CHANGEOS_DECODER_CHANNELS}"
        )

    print(
        "\n✅ Checkpoint metadata is consistent "
        "with ChangeOS-R18."
    )


# ============================================================
# 5. LOAD BEST CHANGEOS-R18 MODEL
# ============================================================

def load_best_changeos_model(
    checkpoint_path,
):
    if "ChangeOSR18" not in globals():
        raise NameError(
            "ChangeOSR18 architecture is not defined. "
            "Run the ChangeOS-R18 architecture cell first."
        )

    checkpoint_path = Path(
        checkpoint_path
    )

    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"ChangeOS checkpoint not found:\n"
            f"{checkpoint_path}"
        )

    print("=" * 72)
    print("LOADING BEST CHANGEOS-R18 CHECKPOINT")
    print("=" * 72)
    print("Checkpoint:", checkpoint_path)
    print("Device    :", DEVICE)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    print_checkpoint_metadata(
        checkpoint
    )

    verify_changeos_checkpoint(
        checkpoint
    )

    # Must exactly match the training-time architecture
    model = ChangeOSR18(
        damage_classes=NUM_DAMAGE_CLASSES,
        imagenet_pretrained=IMAGENET_PRETRAINED,
        input_is_bgr=INPUT_IS_BGR,
        decoder_channels=(
            CHANGEOS_DECODER_CHANNELS
        ),
    ).to(
        DEVICE
    )

    state_dict = extract_model_state_dict(
        checkpoint
    )

    state_dict = remove_module_prefix(
        state_dict
    )

    incompatible_keys = model.load_state_dict(
        state_dict,
        strict=True,
    )

    print("\nCheckpoint loading result:")
    print(
        "Missing keys   :",
        incompatible_keys.missing_keys,
    )
    print(
        "Unexpected keys:",
        incompatible_keys.unexpected_keys,
    )

    if incompatible_keys.missing_keys:
        raise RuntimeError(
            "Missing model keys were detected."
        )

    if incompatible_keys.unexpected_keys:
        raise RuntimeError(
            "Unexpected model keys were detected."
        )

    model.eval()

    print("\n✅ Best ChangeOS-R18 model loaded successfully.")
    print("Checkpoint:", checkpoint_path)
    print("Device    :", DEVICE)
    print("=" * 72)

    return (
        model,
        checkpoint,
    )


# ============================================================
# 6. EXPLICIT CHECKPOINT PATH
# ============================================================

# ============================================================
# SELECT AND LOAD ACTUAL BEST CHANGEOS-R18 CHECKPOINT
# ============================================================

from pathlib import Path


CHANGEOS_TEST_CHECKPOINT = Path(
    "/content/drive/MyDrive/STRATA_runs/"
    "PrePost_Baselines/"
    "ChangeOS_R18/"
    "budget500_seed42/"
    "models/"
    "damage/"
    "best_changeos_r18.pth"
)

if not CHANGEOS_TEST_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Checkpoint not found:\n"
        f"{CHANGEOS_TEST_CHECKPOINT}"
    )

BEST_MODEL_PATH = CHANGEOS_TEST_CHECKPOINT

print("Selected ChangeOS checkpoint:")
print(CHANGEOS_TEST_CHECKPOINT)
print(
    "Size:",
    f"{CHANGEOS_TEST_CHECKPOINT.stat().st_size / 1024**2:.2f} MB",
)

(
    changeos_test_model,
    changeos_checkpoint,
) = load_best_changeos_model(
    checkpoint_path=CHANGEOS_TEST_CHECKPOINT
)

Selected ChangeOS checkpoint:
/content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models/damage/best_changeos_r18.pth
Size: 289.33 MB
LOADING BEST CHANGEOS-R18 CHECKPOINT
Checkpoint: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models/damage/best_changeos_r18.pth
Device    : cuda

Checkpoint metadata:
  model_name: ChangeOS_R18
  run_name: budget500_seed42
  epoch: 48
  best_score: 0.181571342084405
  selection_metric: miou_rare_2class
  num_damage_classes: 4
  decoder_channels: 256
  imagenet_pretrained: False
  input_is_bgr: True

✅ Checkpoint metadata is consistent with ChangeOS-R18.
Model state dictionary found in checkpoint['model'].

Checkpoint loading result:
Missing keys   : []
Unexpected keys: []

✅ Best ChangeOS-R18 model loaded successfully.
Checkpoint: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/models/damage/best_changeos_r18.pth
Device    : cuda


In [45]:
# ============================================================
# CHANGEOS-R18 DORIAN TEST:
# PREDICTION + METRICS + EXCEL + VIOLIN CHARTS
# ============================================================

import os
import re
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
from scipy import ndimage
from torch.utils.data import Dataset, DataLoader
from matplotlib import font_manager as fm
from matplotlib import cm as mpl_cm
from matplotlib import colors as mpl_colors


# ============================================================
# 1. SETTINGS
# ============================================================

EVENT_NAME = "HURRICANE-DORIAN"

EVENT_TAG = re.sub(
    r"[^A-Za-z0-9]+",
    "_",
    EVENT_NAME,
).strip("_")

MODEL_NAME = "ChangeOS_R18"
RUN_NAME = "budget500_seed42"

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

NUM_CLASSES = 4

CLASS_NAMES = [
    "background",
    "intact",
    "damage",
    "destroyed",
]

FOREGROUND_CLASS_NAMES = [
    "intact",
    "damage",
    "destroyed",
]

PAIRED_TEST_ROOT = Path(
    "/content/ebd_HURRICANE-DORIAN_local/"
    "HURRICANE-DORIAN_prepost_npz_512valid_as_256"
)

TEST_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/STRATA_runs/"
    "PrePost_Baselines/"
    "ChangeOS_R18/"
    "budget500_seed42/"
    "HURRICANE_DORIAN_test_metrics"
)

TEST_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXCEL_PATH = (
    TEST_OUTPUT_DIR /
    "HURRICANE_DORIAN_ChangeOS_R18_test_metrics.xlsx"
)

SUMMARY_TEXT_PATH = (
    TEST_OUTPUT_DIR /
    "HURRICANE_DORIAN_ChangeOS_R18_test_summary.txt"
)

SUMMARY_JSON_PATH = (
    TEST_OUTPUT_DIR /
    "HURRICANE_DORIAN_ChangeOS_R18_test_summary.json"
)

LEGACY_PIXEL_PNG = (
    TEST_OUTPUT_DIR /
    "HURRICANE_DORIAN_ChangeOS_R18_pixelwise_violin.png"
)

LEGACY_PIXEL_PDF = (
    TEST_OUTPUT_DIR /
    "HURRICANE_DORIAN_ChangeOS_R18_pixelwise_violin.pdf"
)

LEGACY_BUILDING_PNG = (
    TEST_OUTPUT_DIR /
    "HURRICANE_DORIAN_ChangeOS_R18_buildingwise_violin.png"
)

LEGACY_BUILDING_PDF = (
    TEST_OUTPUT_DIR /
    "HURRICANE_DORIAN_ChangeOS_R18_buildingwise_violin.pdf"
)

STRICT_PIXEL_PNG = (
    TEST_OUTPUT_DIR /
    "HURRICANE_DORIAN_ChangeOS_R18_strict_pixelwise_violin.png"
)

STRICT_PIXEL_PDF = (
    TEST_OUTPUT_DIR /
    "HURRICANE_DORIAN_ChangeOS_R18_strict_pixelwise_violin.pdf"
)

STRICT_BUILDING_PNG = (
    TEST_OUTPUT_DIR /
    "HURRICANE_DORIAN_ChangeOS_R18_strict_buildingwise_violin.png"
)

STRICT_BUILDING_PDF = (
    TEST_OUTPUT_DIR /
    "HURRICANE_DORIAN_ChangeOS_R18_strict_buildingwise_violin.pdf"
)

TEST_BATCH_SIZE = 16
TEST_NUM_WORKERS = 0
FIGURE_DPI = 800

# Original ChangeOS weights:
# no-damage=8, minor=38, major=25, destroyed=11
#
# Because no-damage and minor are merged into Intact,
# the larger merged-class weight is retained.
OBJECT_VOTE_WEIGHTS = {
    1: 38.0,  # Intact
    2: 25.0,  # Damage
    3: 11.0,  # Destroyed
}


# ============================================================
# 2. TEST DATASET
# ============================================================

class ChangeOSDorianTestDataset(Dataset):

    def __init__(self, root_dir):
        self.root_dir = Path(root_dir)

        self.files = sorted(
            self.root_dir.glob("*.npz")
        )

        if len(self.files) == 0:
            raise FileNotFoundError(
                f"No test NPZ files found in:\n"
                f"{self.root_dir}"
            )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        file_path = self.files[index]

        with np.load(file_path) as data:
            pre_image = np.array(
                data["pre_image"],
                dtype=np.uint8,
                copy=True,
            )

            post_image = np.array(
                data["post_image"],
                dtype=np.uint8,
                copy=True,
            )

            damage_mask = np.array(
                data["damage_mask"],
                dtype=np.uint8,
                copy=True,
            )

            building_mask = np.array(
                data["building_mask"],
                dtype=np.uint8,
                copy=True,
            )

        # Same normalization used during training.
        pre_image = (
            pre_image.astype(np.float32)
            / 255.0
        )

        post_image = (
            post_image.astype(np.float32)
            / 255.0
        )

        pre_image = np.ascontiguousarray(
            np.transpose(
                pre_image,
                (2, 0, 1),
            )
        )

        post_image = np.ascontiguousarray(
            np.transpose(
                post_image,
                (2, 0, 1),
            )
        )

        damage_mask = np.ascontiguousarray(
            damage_mask
        )

        building_mask = np.ascontiguousarray(
            building_mask
        )

        return (
            torch.from_numpy(
                pre_image
            ).float(),

            torch.from_numpy(
                post_image
            ).float(),

            torch.from_numpy(
                damage_mask
            ).long(),

            torch.from_numpy(
                building_mask
            ).float(),

            EVENT_NAME,

            str(file_path),
        )


test_dataset = ChangeOSDorianTestDataset(
    PAIRED_TEST_ROOT
)

test_loader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=TEST_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)


# ============================================================
# 3. FILENAME PARSER
# ============================================================

def parse_patch_filename(path):
    stem = Path(path).stem

    try:
        base_name, y_string, x_string = (
            stem.rsplit("_", 2)
        )

        return (
            base_name,
            int(y_string),
            int(x_string),
        )

    except Exception:
        return (
            stem,
            np.nan,
            np.nan,
        )


# ============================================================
# 4. CHANGEOS OBJECT-BASED INFERENCE
# ============================================================

def changeos_object_vote(
    localization_mask,
    raw_damage_prediction,
):
    """
    Four-class adaptation of official ChangeOS object voting.

    localization_mask:
        binary prediction from localization_logits > 0

    raw_damage_prediction:
        argmax of four-class damage logits
    """
    localization_mask = (
        localization_mask > 0
    )

    structure = np.ones(
        (3, 3),
        dtype=np.uint8,
    )

    labeled_objects, number_of_objects = (
        ndimage.label(
            localization_mask,
            structure=structure,
        )
    )

    refined_damage = np.zeros_like(
        raw_damage_prediction,
        dtype=np.uint8,
    )

    for object_id in range(
        1,
        number_of_objects + 1,
    ):
        object_region = (
            labeled_objects == object_id
        )

        weighted_votes = []

        for damage_class in [
            1,
            2,
            3,
        ]:
            pixel_count = np.sum(
                raw_damage_prediction[
                    object_region
                ] == damage_class
            )

            weighted_vote = (
                float(pixel_count)
                * OBJECT_VOTE_WEIGHTS[
                    damage_class
                ]
            )

            weighted_votes.append(
                weighted_vote
            )

        winning_class = (
            int(np.argmax(weighted_votes))
            + 1
        )

        refined_damage[
            object_region
        ] = winning_class

    return refined_damage


# ============================================================
# 5. BUILDING-MAJORITY MAP
# ============================================================

def building_majority_map(
    label_map,
    building_mask,
):
    building_binary = (
        building_mask > 0
    )

    labeled_buildings, number_of_buildings = (
        ndimage.label(
            building_binary
        )
    )

    output = np.zeros_like(
        label_map,
        dtype=np.uint8,
    )

    for building_id in range(
        1,
        number_of_buildings + 1,
    ):
        building_region = (
            labeled_buildings == building_id
        )

        class_values = label_map[
            building_region
        ]

        class_values = class_values[
            class_values > 0
        ]

        if class_values.size == 0:
            majority_class = 0

        else:
            majority_class = np.bincount(
                class_values.astype(np.int64),
                minlength=NUM_CLASSES,
            ).argmax()

        output[
            building_region
        ] = majority_class

    return output


# ============================================================
# 6. CONFUSION MATRIX
# ============================================================

def confusion_matrix_numpy(
    prediction,
    target,
    valid_mask,
):
    prediction = np.asarray(
        prediction
    ).reshape(-1)

    target = np.asarray(
        target
    ).reshape(-1)

    valid_mask = np.asarray(
        valid_mask
    ).reshape(-1).astype(bool)

    valid = (
        valid_mask
        & (target >= 0)
        & (target < NUM_CLASSES)
        & (prediction >= 0)
        & (prediction < NUM_CLASSES)
    )

    prediction = prediction[
        valid
    ].astype(np.int64)

    target = target[
        valid
    ].astype(np.int64)

    if target.size == 0:
        return np.zeros(
            (NUM_CLASSES, NUM_CLASSES),
            dtype=np.float64,
        )

    return np.bincount(
        NUM_CLASSES * target
        + prediction,
        minlength=NUM_CLASSES ** 2,
    ).reshape(
        NUM_CLASSES,
        NUM_CLASSES,
    ).astype(
        np.float64
    )


# ============================================================
# 7. LEGACY STRATA METRICS
# ============================================================

def compute_legacy_metrics(matrix):
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    evaluated = matrix[1:, 1:]

    true_positive = np.diag(
        evaluated
    )

    false_positive = (
        evaluated.sum(axis=0)
        - true_positive
    )

    false_negative = (
        evaluated.sum(axis=1)
        - true_positive
    )

    total = evaluated.sum()

    accuracy = (
        true_positive.sum()
        / (total + 1e-6)
    )

    precision = (
        true_positive
        / (
            true_positive
            + false_positive
            + 1e-6
        )
    )

    recall = (
        true_positive
        / (
            true_positive
            + false_negative
            + 1e-6
        )
    )

    f1 = (
        2 * precision * recall
        / (
            precision
            + recall
            + 1e-6
        )
    )

    iou = (
        true_positive
        / (
            true_positive
            + false_positive
            + false_negative
            + 1e-6
        )
    )

    expected = (
        (
            evaluated.sum(axis=0)
            * evaluated.sum(axis=1)
        ).sum()
        / ((total ** 2) + 1e-6)
    )

    kappa = (
        accuracy - expected
    ) / (
        1 - expected + 1e-6
    )

    return make_metric_dictionary(
        accuracy,
        precision,
        recall,
        f1,
        iou,
        kappa,
    )


# ============================================================
# 8. STRICT METRICS
# ============================================================

def compute_strict_metrics(matrix):
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    classes = [
        1,
        2,
        3,
    ]

    true_positive = np.array([
        matrix[class_id, class_id]
        for class_id in classes
    ])

    false_positive = np.array([
        matrix[:, class_id].sum()
        - matrix[class_id, class_id]
        for class_id in classes
    ])

    false_negative = np.array([
        matrix[class_id, :].sum()
        - matrix[class_id, class_id]
        for class_id in classes
    ])

    precision = (
        true_positive
        / (
            true_positive
            + false_positive
            + 1e-6
        )
    )

    recall = (
        true_positive
        / (
            true_positive
            + false_negative
            + 1e-6
        )
    )

    f1 = (
        2 * precision * recall
        / (
            precision
            + recall
            + 1e-6
        )
    )

    iou = (
        true_positive
        / (
            true_positive
            + false_positive
            + false_negative
            + 1e-6
        )
    )

    strict_matrix = matrix.copy()
    strict_matrix[0, :] = 0

    total = strict_matrix.sum()

    accuracy = (
        true_positive.sum()
        / (total + 1e-6)
    )

    expected = (
        (
            strict_matrix.sum(axis=0)
            * strict_matrix.sum(axis=1)
        ).sum()
        / ((total ** 2) + 1e-6)
    )

    kappa = (
        accuracy - expected
    ) / (
        1 - expected + 1e-6
    )

    return make_metric_dictionary(
        accuracy,
        precision,
        recall,
        f1,
        iou,
        kappa,
    )


def make_metric_dictionary(
    accuracy,
    precision,
    recall,
    f1,
    iou,
    kappa,
):
    metrics = {
        "accuracy": float(accuracy),
        "precision_macro": float(
            np.mean(precision)
        ),
        "recall_macro": float(
            np.mean(recall)
        ),
        "f1_macro": float(
            np.mean(f1)
        ),
        "kappa": float(kappa),
        "miou_3class": float(
            np.mean(iou)
        ),
        "miou_rare_2class": float(
            np.mean(iou[1:3])
        ),
    }

    for index, class_name in enumerate(
        FOREGROUND_CLASS_NAMES
    ):
        metrics[
            f"precision_{class_name}"
        ] = float(precision[index])

        metrics[
            f"recall_{class_name}"
        ] = float(recall[index])

        metrics[
            f"f1_{class_name}"
        ] = float(f1[index])

        metrics[
            f"iou_{class_name}"
        ] = float(iou[index])

    return metrics


# ============================================================
# 9. PATCH ROW
# ============================================================

def make_patch_row(
    file_path,
    mode,
    metrics,
    valid_pixels,
):
    base_name, y, x = (
        parse_patch_filename(
            file_path
        )
    )

    return {
        "file": Path(file_path).name,
        "base_name": base_name,
        "y": y,
        "x": x,
        "event": EVENT_NAME,
        "valid_building_pixels": int(
            valid_pixels
        ),
        "mode": mode,
        **metrics,
    }


# ============================================================
# 10. COMPLETE PREDICTION
# ============================================================

def run_changeos_test():
    pixel_matrix = np.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        dtype=np.float64,
    )

    building_matrix = np.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        dtype=np.float64,
    )

    legacy_pixel_rows = []
    legacy_building_rows = []
    strict_pixel_rows = []
    strict_building_rows = []

    target_counts = Counter()
    predicted_counts = Counter()

    changeos_test_model.eval()

    with torch.inference_mode():
        for batch in tqdm(
            test_loader,
            desc="ChangeOS Dorian prediction",
        ):
            (
                pre_images,
                post_images,
                damage_targets,
                building_masks,
                event_names,
                file_paths,
            ) = batch

            pre_images = pre_images.to(
                DEVICE,
                non_blocking=True,
            )

            post_images = post_images.to(
                DEVICE,
                non_blocking=True,
            )

            (
                localization_logits,
                damage_logits,
            ) = changeos_test_model(
                pre_images,
                post_images,
            )

            localization_predictions = (
                localization_logits[:, 0]
                > 0.0
            ).cpu().numpy()

            raw_damage_predictions = (
                torch.argmax(
                    damage_logits,
                    dim=1,
                )
                .cpu()
                .numpy()
                .astype(np.uint8)
            )

            damage_targets = (
                damage_targets
                .numpy()
                .astype(np.uint8)
            )

            building_masks = (
                building_masks
                .numpy()
                .astype(np.uint8)
            )

            for index in range(
                len(file_paths)
            ):
                prediction = (
                    changeos_object_vote(
                        localization_predictions[
                            index
                        ],
                        raw_damage_predictions[
                            index
                        ],
                    )
                )

                target = damage_targets[
                    index
                ]

                building = building_masks[
                    index
                ]

                valid_mask = (
                    building > 0
                )

                patch_pixel_matrix = (
                    confusion_matrix_numpy(
                        prediction,
                        target,
                        valid_mask,
                    )
                )

                pixel_matrix += (
                    patch_pixel_matrix
                )

                target_building_map = (
                    building_majority_map(
                        target,
                        building,
                    )
                )

                prediction_building_map = (
                    building_majority_map(
                        prediction,
                        building,
                    )
                )

                patch_building_matrix = (
                    confusion_matrix_numpy(
                        prediction_building_map,
                        target_building_map,
                        valid_mask,
                    )
                )

                building_matrix += (
                    patch_building_matrix
                )

                legacy_pixel_rows.append(
                    make_patch_row(
                        file_paths[index],
                        "pixel_wise",
                        compute_legacy_metrics(
                            patch_pixel_matrix
                        ),
                        valid_mask.sum(),
                    )
                )

                strict_pixel_rows.append(
                    make_patch_row(
                        file_paths[index],
                        "strict_pixel_wise",
                        compute_strict_metrics(
                            patch_pixel_matrix
                        ),
                        valid_mask.sum(),
                    )
                )

                legacy_building_rows.append(
                    make_patch_row(
                        file_paths[index],
                        "building_wise_majority",
                        compute_legacy_metrics(
                            patch_building_matrix
                        ),
                        valid_mask.sum(),
                    )
                )

                strict_building_rows.append(
                    make_patch_row(
                        file_paths[index],
                        "strict_building_majority",
                        compute_strict_metrics(
                            patch_building_matrix
                        ),
                        valid_mask.sum(),
                    )
                )

                for class_id, count in zip(
                    *np.unique(
                        target[valid_mask],
                        return_counts=True,
                    )
                ):
                    target_counts[
                        int(class_id)
                    ] += int(count)

                for class_id, count in zip(
                    *np.unique(
                        prediction[valid_mask],
                        return_counts=True,
                    )
                ):
                    predicted_counts[
                        int(class_id)
                    ] += int(count)

    return {
        "legacy_global_pixel":
            compute_legacy_metrics(
                pixel_matrix
            ),

        "legacy_global_building":
            compute_legacy_metrics(
                building_matrix
            ),

        "strict_global_pixel":
            compute_strict_metrics(
                pixel_matrix
            ),

        "strict_global_building":
            compute_strict_metrics(
                building_matrix
            ),

        "legacy_patch_pixel_df":
            pd.DataFrame(
                legacy_pixel_rows
            ),

        "legacy_patch_building_df":
            pd.DataFrame(
                legacy_building_rows
            ),

        "strict_patch_pixel_df":
            pd.DataFrame(
                strict_pixel_rows
            ),

        "strict_patch_building_df":
            pd.DataFrame(
                strict_building_rows
            ),

        "pixel_confusion":
            pixel_matrix,

        "building_confusion":
            building_matrix,

        "target_class_counts":
            dict(target_counts),

        "predicted_class_counts":
            dict(predicted_counts),
    }


test_results = run_changeos_test()


# ============================================================
# 11. SAVE EXCEL
# ============================================================

def make_global_dataframe(
    metrics,
    mode,
    protocol,
):
    return pd.DataFrame([{
        "event": EVENT_NAME,
        "model": MODEL_NAME,
        "run": RUN_NAME,
        "mode": mode,
        "protocol": protocol,
        **metrics,
    }])


legacy_global_pixel_df = make_global_dataframe(
    test_results[
        "legacy_global_pixel"
    ],
    "pixel_wise_global",
    "legacy_STRATA",
)

legacy_global_building_df = make_global_dataframe(
    test_results[
        "legacy_global_building"
    ],
    "building_wise_majority_global",
    "legacy_STRATA",
)

strict_global_pixel_df = make_global_dataframe(
    test_results[
        "strict_global_pixel"
    ],
    "pixel_wise_global",
    "strict",
)

strict_global_building_df = make_global_dataframe(
    test_results[
        "strict_global_building"
    ],
    "building_wise_majority_global",
    "strict",
)

pixel_confusion_df = pd.DataFrame(
    test_results[
        "pixel_confusion"
    ],
    index=[
        f"true_{name}"
        for name in CLASS_NAMES
    ],
    columns=[
        f"pred_{name}"
        for name in CLASS_NAMES
    ],
)

building_confusion_df = pd.DataFrame(
    test_results[
        "building_confusion"
    ],
    index=[
        f"true_{name}"
        for name in CLASS_NAMES
    ],
    columns=[
        f"pred_{name}"
        for name in CLASS_NAMES
    ],
)

configuration_df = pd.DataFrame([
    {
        "parameter": "event",
        "value": EVENT_NAME,
    },
    {
        "parameter": "model",
        "value": MODEL_NAME,
    },
    {
        "parameter": "run",
        "value": RUN_NAME,
    },
    {
        "parameter": "test_patches",
        "value": len(test_dataset),
    },
    {
        "parameter": "complete_512_blocks",
        "value": len(test_dataset) // 4,
    },
    {
        "parameter": "checkpoint",
        "value": str(
            CHANGEOS_TEST_CHECKPOINT
        ),
    },
    {
        "parameter": "inference",
        "value": "ChangeOS object-based inference",
    },
    {
        "parameter": "object_vote_weights",
        "value": str(
            OBJECT_VOTE_WEIGHTS
        ),
    },
])

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl",
) as writer:
    legacy_global_pixel_df.to_excel(
        writer,
        sheet_name="global_pixel_metrics",
        index=False,
    )

    legacy_global_building_df.to_excel(
        writer,
        sheet_name="global_building_metrics",
        index=False,
    )

    test_results[
        "legacy_patch_pixel_df"
    ].to_excel(
        writer,
        sheet_name="patch_pixel_metrics",
        index=False,
    )

    test_results[
        "legacy_patch_building_df"
    ].to_excel(
        writer,
        sheet_name="patch_building_metrics",
        index=False,
    )

    pixel_confusion_df.to_excel(
        writer,
        sheet_name="cm_pixel",
    )

    building_confusion_df.to_excel(
        writer,
        sheet_name="cm_building",
    )

    strict_global_pixel_df.to_excel(
        writer,
        sheet_name="strict_global_pixel",
        index=False,
    )

    strict_global_building_df.to_excel(
        writer,
        sheet_name="strict_global_building",
        index=False,
    )

    test_results[
        "strict_patch_pixel_df"
    ].to_excel(
        writer,
        sheet_name="strict_patch_pixel",
        index=False,
    )

    test_results[
        "strict_patch_building_df"
    ].to_excel(
        writer,
        sheet_name="strict_patch_building",
        index=False,
    )

    configuration_df.to_excel(
        writer,
        sheet_name="configuration",
        index=False,
    )


# ============================================================
# 12. SAVE TEXT AND JSON
# ============================================================

summary = {
    "event": EVENT_NAME,
    "model": MODEL_NAME,
    "run": RUN_NAME,
    "checkpoint": str(
        CHANGEOS_TEST_CHECKPOINT
    ),
    "test_patches": len(
        test_dataset
    ),
    "complete_512_blocks": (
        len(test_dataset) // 4
    ),
    "legacy_global_pixel":
        test_results[
            "legacy_global_pixel"
        ],
    "legacy_global_building":
        test_results[
            "legacy_global_building"
        ],
    "strict_global_pixel":
        test_results[
            "strict_global_pixel"
        ],
    "strict_global_building":
        test_results[
            "strict_global_building"
        ],
    "target_class_counts":
        test_results[
            "target_class_counts"
        ],
    "predicted_class_counts":
        test_results[
            "predicted_class_counts"
        ],
}

with open(
    SUMMARY_JSON_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        summary,
        output_file,
        indent=4,
    )

with open(
    SUMMARY_TEXT_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    output_file.write(
        "ChangeOS-R18 Independent Dorian Test\n"
    )

    output_file.write(
        "=" * 60 + "\n"
    )

    for section_name in [
        "legacy_global_pixel",
        "legacy_global_building",
        "strict_global_pixel",
        "strict_global_building",
    ]:
        output_file.write(
            f"\n{section_name.upper()}\n"
        )

        for metric_name, metric_value in (
            summary[
                section_name
            ].items()
        ):
            output_file.write(
                f"{metric_name}: "
                f"{metric_value:.6f}\n"
            )


# ============================================================
# 13. VIOLIN PLOT
# ============================================================

def get_serif_font():
    available = {
        font.name
        for font in fm.fontManager.ttflist
    }

    for font_name in [
        "Times New Roman",
        "Liberation Serif",
        "Nimbus Roman",
        "DejaVu Serif",
    ]:
        if font_name in available:
            return font_name

    return "serif"


def save_violin(
    dataframe,
    png_path,
    pdf_path,
):
    metric_columns = [
        ("accuracy", "Accuracy"),
        ("f1_macro", "F-score"),
        ("miou_3class", "mIoU"),
    ]

    values_list = []
    labels = []

    for column, label in metric_columns:
        values = pd.to_numeric(
            dataframe[column],
            errors="coerce",
        ).dropna().to_numpy()

        values = values[
            np.isfinite(values)
        ]

        if len(values) == 1:
            values = np.repeat(
                values,
                2,
            )

        values_list.append(
            values
        )

        labels.append(
            label
        )

    plt.rcParams["font.family"] = (
        get_serif_font()
    )

    plt.rcParams["pdf.fonttype"] = 42

    figure, axis = plt.subplots(
        figsize=(6.4, 4.4)
    )

    violin = axis.violinplot(
        values_list,
        showmeans=False,
        showmedians=False,
        showextrema=False,
        widths=0.78,
    )

    normalization = mpl_colors.Normalize(
        0.0,
        1.0,
    )

    color_map = mpl_cm.get_cmap(
        "jet"
    )

    for index, body in enumerate(
        violin["bodies"]
    ):
        mean_value = np.mean(
            values_list[index]
        )

        body.set_facecolor(
            color_map(
                normalization(
                    mean_value
                )
            )
        )

        body.set_edgecolor(
            "black"
        )

        body.set_alpha(
            0.45
        )

    random_generator = (
        np.random.default_rng(42)
    )

    for position, values in enumerate(
        values_list,
        start=1,
    ):
        jitter = random_generator.normal(
            position,
            0.045,
            len(values),
        )

        axis.scatter(
            jitter,
            values,
            c=values,
            cmap="jet",
            vmin=0,
            vmax=1,
            s=13,
            alpha=0.72,
            edgecolors="none",
        )

        axis.plot(
            [
                position - 0.22,
                position + 0.22,
            ],
            [
                np.median(values),
                np.median(values),
            ],
            color="black",
            linewidth=1.4,
        )

        axis.scatter(
            position,
            np.mean(values),
            marker="D",
            s=38,
            color="black",
        )

    axis.set_xticks(
        range(
            1,
            len(labels) + 1,
        )
    )

    axis.set_xticklabels(
        labels
    )

    axis.set_ylabel(
        "Patch-level score"
    )

    axis.set_ylim(
        0,
        1.02,
    )

    axis.grid(
        axis="y",
        linestyle="--",
        linewidth=0.5,
        alpha=0.45,
    )

    axis.spines[
        "top"
    ].set_visible(False)

    axis.spines[
        "right"
    ].set_visible(False)

    scalar_mappable = mpl_cm.ScalarMappable(
        cmap="jet",
        norm=normalization,
    )

    scalar_mappable.set_array([])

    color_bar = figure.colorbar(
        scalar_mappable,
        ax=axis,
        fraction=0.045,
        pad=0.035,
    )

    color_bar.set_label(
        "Score value"
    )

    figure.tight_layout()

    figure.savefig(
        png_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )

    figure.savefig(
        pdf_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )


save_violin(
    test_results[
        "legacy_patch_pixel_df"
    ],
    LEGACY_PIXEL_PNG,
    LEGACY_PIXEL_PDF,
)

save_violin(
    test_results[
        "legacy_patch_building_df"
    ],
    LEGACY_BUILDING_PNG,
    LEGACY_BUILDING_PDF,
)

save_violin(
    test_results[
        "strict_patch_pixel_df"
    ],
    STRICT_PIXEL_PNG,
    STRICT_PIXEL_PDF,
)

save_violin(
    test_results[
        "strict_patch_building_df"
    ],
    STRICT_BUILDING_PNG,
    STRICT_BUILDING_PDF,
)


# ============================================================
# 14. FINAL RESULT DISPLAY
# ============================================================

comparison_df = pd.DataFrame([
    {
        "Protocol": "Legacy_STRATA",
        "Mode": "Pixel-wise",
        **test_results[
            "legacy_global_pixel"
        ],
    },
    {
        "Protocol": "Legacy_STRATA",
        "Mode": "Building-majority",
        **test_results[
            "legacy_global_building"
        ],
    },
    {
        "Protocol": "Strict",
        "Mode": "Pixel-wise",
        **test_results[
            "strict_global_pixel"
        ],
    },
    {
        "Protocol": "Strict",
        "Mode": "Building-majority",
        **test_results[
            "strict_global_building"
        ],
    },
])

display(
    comparison_df.round(4)
)

print("\n" + "=" * 72)
print("CHANGEOS-R18 DORIAN TEST COMPLETED")
print("=" * 72)
print("Test patches:", len(test_dataset))
print("Excel       :", EXCEL_PATH)
print("Summary TXT :", SUMMARY_TEXT_PATH)
print("Summary JSON:", SUMMARY_JSON_PATH)
print("Charts      :", TEST_OUTPUT_DIR)
print("=" * 72)

ChangeOS Dorian prediction:   0%|          | 0/46 [00:00<?, ?it/s]

/tmp/ipykernel_1071/1604373387.py:1401: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  color_map = mpl_cm.get_cmap(
/tmp/ipykernel_1071/1604373387.py:1401: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  color_map = mpl_cm.get_cmap(
/tmp/ipykernel_1071/1604373387.py:1401: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  color_map = mpl_cm.get_cmap(
/tmp/ipykernel_1071/1604373387.py:1401: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 

,Protocol,Mode,accuracy,precision_macro,recall_macro,f1_macro,kappa,miou_3class,miou_rare_2class,precision_intact,...,f1_intact,iou_intact,precision_damage,recall_damage,f1_damage,iou_damage,precision_destroyed,recall_destroyed,f1_destroyed,iou_destroyed
0,Legacy_STRATA,Pixel-wise,0.5892,0.6425,0.5347,0.5548,0.3132,0.3873,0.3831,0.6047,...,0.5669,0.3955,0.5574,0.7403,0.6360,0.4662,0.7655,0.3304,0.4616,0.3001
1,Legacy_STRATA,Building-majority,0.5918,0.6458,0.5378,0.5580,0.3181,0.3903,0.3853,0.6038,...,0.5718,0.4004,0.5607,0.7376,0.6371,0.4674,0.7728,0.3328,0.4652,0.3031
2,Strict,Pixel-wise,0.5663,0.6425,0.5139,0.5420,0.2950,0.3754,0.3714,0.6047,...,0.5542,0.3833,0.5574,0.7169,0.6272,0.4568,0.7655,0.3134,0.4447,0.2859
3,Strict,Building-majority,0.5866,0.6458,0.5332,0.5552,0.3142,0.3877,0.3836,0.6038,...,0.5671,0.3957,0.5607,0.7352,0.6362,0.4665,0.7728,0.3299,0.4624,0.3007



CHANGEOS-R18 DORIAN TEST COMPLETED
Test patches: 724
Excel       : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_ChangeOS_R18_test_metrics.xlsx
Summary TXT : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_ChangeOS_R18_test_summary.txt
Summary JSON: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_ChangeOS_R18_test_summary.json
Charts      : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18/budget500_seed42/HURRICANE_DORIAN_test_metrics
